In [6]:
"""
Theta-Band → Clustering: Between-Subject Analysis
==================================================

Identical pipeline to the gamma between-subject script, but targeting
the theta band (4–8 Hz) instead of gamma (≥40 Hz).

Approach:
  1. Filter to theta band (4–8 Hz), average over frequencies per trial
     per subject × region → one value per subject per region
  2. Average clustering (SP & T separately) per subject
  3. Stack SP & T into long format
  4. Fit LMM (n_subjects >= 15) or OLS + HC3 robust SEs:

     clustering_mean ~ theta_mean * clustering_type * region
     + (1 | subject)

  Separate models for OSC and FRAC components.
"""

import warnings
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats

warnings.filterwarnings('ignore')

# ============================================================================
# CONFIG
# ============================================================================

DATA_FILE  = 'ALL_SUBJECTS_irasa_clean.csv'
PHASE      = 'retrieval'
THETA_LO   = 8    # Hz
THETA_HI   = 16    # Hz

# ============================================================================
# LOAD & CLEAN
# ============================================================================

df = pd.read_csv(DATA_FILE)
df.replace(-999, np.nan, inplace=True)

print(f"Loaded {len(df):,} rows | {df['subject'].nunique()} subjects")

df['subj_session'] = df['subject'] + '_' + df['session'].astype(str)
df['region']       = pd.Categorical(df['region'], categories=['RHP', 'LHP'])

# ============================================================================
# THETA BAND-AVERAGE PER TRIAL (collapse frequencies first)
# ============================================================================

osc_col  = f'{PHASE}_osc_ssl'
frac_col = f'{PHASE}_frac_ssl'

GROUP_KEYS = ['subject', 'session', 'trial', 'recalled_word', 'region']

theta_mask = (df['freq_hz'] >= THETA_LO) & (df['freq_hz'] <= THETA_HI)
print(f"Theta-band rows ({THETA_LO}–{THETA_HI} Hz): {theta_mask.sum():,}")

theta_df = (
    df[theta_mask]
    .groupby(GROUP_KEYS, as_index=False, observed=True)
    .agg({
        osc_col:                   'mean',
        frac_col:                  'mean',
        'subj_session':            'first',
        'SP_clustering_from_prev': 'first',
        'T_clustering_from_prev':  'first',
    })
)

print(f"Trial-level theta rows: {len(theta_df):,} | Subjects: {theta_df['subject'].nunique()}\n")

# ============================================================================
# AVERAGE TO SUBJECT LEVEL  (per subject × region)
# ============================================================================

subj_df = (
    theta_df
    .groupby(['subject', 'region'], as_index=False, observed=True)
    .agg(
        theta_osc_mean  = (osc_col,                   'mean'),
        theta_frac_mean = (frac_col,                  'mean'),
        SP_clust_mean   = ('SP_clustering_from_prev', 'mean'),
        T_clust_mean    = ('T_clustering_from_prev',  'mean'),
        n_trials        = (osc_col,                   'count'),
    )
)

print(f"Subject-level rows (subject × region): {len(subj_df):,}")
print(f"  Subjects: {subj_df['subject'].nunique()}")
print(f"  Trials per subject (mean ± SD): "
      f"{subj_df['n_trials'].mean():.1f} ± {subj_df['n_trials'].std():.1f}\n")
print(subj_df[['subject', 'region', 'n_trials',
               'theta_osc_mean', 'theta_frac_mean',
               'SP_clust_mean', 'T_clust_mean']].to_string(index=False))
print()

# ============================================================================
# STACK SP & T → LONG FORMAT
# ============================================================================

sp                    = subj_df.copy()
sp['clustering_mean'] = sp['SP_clust_mean']
sp['clustering_type'] = 'spatial'

tc                    = subj_df.copy()
tc['clustering_mean'] = tc['T_clust_mean']
tc['clustering_type'] = 'temporal'

long_df = pd.concat([sp, tc], ignore_index=True)
long_df = long_df.dropna(subset=['clustering_mean', 'theta_osc_mean', 'theta_frac_mean'])

long_df['clustering_type'] = pd.Categorical(
    long_df['clustering_type'], categories=['spatial', 'temporal']
)
long_df['region'] = pd.Categorical(long_df['region'], categories=['RHP', 'LHP'])

print(f"Long-format rows: {len(long_df):,}")
print(f"  spatial : {(long_df['clustering_type']=='spatial').sum():,}")
print(f"  temporal: {(long_df['clustering_type']=='temporal').sum():,}\n")

n_subjects = long_df['subject'].nunique()

# ============================================================================
# PARAMETER LABELS
# ============================================================================

param_labels_osc = {
    'Intercept':                                                    'Intercept',
    'theta_osc_mean':                                               'Theta Osc  (RHP, spatial)',
    'clustering_type[T.temporal]':                                  'Clustering type  (temporal vs spatial)',
    'region[T.LHP]':                                                'Region  (LHP vs RHP)',
    'theta_osc_mean:clustering_type[T.temporal]':                   'Theta Osc × Clustering type',
    'theta_osc_mean:region[T.LHP]':                                 'Theta Osc × Region',
    'clustering_type[T.temporal]:region[T.LHP]':                    'Clustering type × Region',
    'theta_osc_mean:clustering_type[T.temporal]:region[T.LHP]':     'Theta Osc × Clustering type × Region',
}

param_labels_frac = {
    'Intercept':                                                     'Intercept',
    'theta_frac_mean':                                               'Theta Frac  (RHP, spatial)',
    'clustering_type[T.temporal]':                                   'Clustering type  (temporal vs spatial)',
    'region[T.LHP]':                                                 'Region  (LHP vs RHP)',
    'theta_frac_mean:clustering_type[T.temporal]':                   'Theta Frac × Clustering type',
    'theta_frac_mean:region[T.LHP]':                                 'Theta Frac × Region',
    'clustering_type[T.temporal]:region[T.LHP]':                     'Clustering type × Region',
    'theta_frac_mean:clustering_type[T.temporal]:region[T.LHP]':     'Theta Frac × Clustering type × Region',
}

# ============================================================================
# MODEL FITTING HELPER
# ============================================================================

def fit_model(data, formula, param_labels, label):
    """
    Fit LMM if n_subjects >= 15, else OLS + HC3 robust SEs.
    Returns summary_rows list.
    """
    print("=" * 80)
    print(f"MODEL: {label}")
    print(f"  {formula}")
    print(f"  n subjects: {data['subject'].nunique()} | n obs: {len(data)}")
    print("=" * 80)

    fit      = None
    fit_type = None

    # ── Try LMM first if enough subjects ────────────────────────────────────
    if n_subjects >= 15:
        try:
            fit = smf.mixedlm(
                formula    = formula,
                data       = data,
                groups     = data['subject'],
                re_formula = '~1',
            ).fit(method='powell', maxiter=2000, disp=False)
            fit_type = 'LMM'
            print(f"  Converged (LMM): {fit.converged}\n")
        except Exception as e:
            print(f"  LMM failed: {e}")

    # ── Fall back to OLS + HC3 ───────────────────────────────────────────────
    if fit is None:
        ols_fit     = smf.ols(formula=formula, data=data).fit()
        robust_fit  = ols_fit.get_robustcov_results(cov_type='HC3')
        param_names = ols_fit.model.exog_names
        fit         = robust_fit
        fit._params  = pd.Series(robust_fit.params,  index=param_names)
        fit._bse     = pd.Series(robust_fit.bse,     index=param_names)
        fit._tvalues = pd.Series(robust_fit.tvalues, index=param_names)
        fit._pvalues = pd.Series(robust_fit.pvalues, index=param_names)
        fit_type     = 'OLS (HC3 robust SE)'
        print(f"  Using {fit_type}\n")

    # ── Unified accessors ────────────────────────────────────────────────────
    def get_params(f):  return f._params   if hasattr(f, '_params')   else f.params
    def get_bse(f):     return f._bse      if hasattr(f, '_bse')      else f.bse
    def get_tvalues(f): return f._tvalues  if hasattr(f, '_tvalues')  else f.tvalues
    def get_pvalues(f): return f._pvalues  if hasattr(f, '_pvalues')  else f.pvalues

    # ── Results table ────────────────────────────────────────────────────────
    print(f"  {'Parameter':<48} {'Coef':>8} {'SE':>8} {'t':>8} {'p':>8}  Sig")
    print("  " + "─" * 78)

    summary_rows = []
    for pname, plabel in param_labels.items():
        if pname not in get_params(fit).index:
            continue
        coef = float(get_params(fit)[pname])
        se   = float(get_bse(fit)[pname])
        t    = float(get_tvalues(fit)[pname])
        p    = float(get_pvalues(fit)[pname])
        sig  = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else ''))
        print(f"  {plabel:<48} {coef:>8.4f} {se:>8.4f} {t:>8.3f} {p:>8.4f}  {sig}")
        summary_rows.append(dict(parameter=plabel, coef=coef, se=se, t=t, p=p, sig=sig,
                                 model=label, fit_type=fit_type))

    print(f"\n  * p<0.05  ** p<0.01  *** p<0.001")

    if fit_type == 'LMM':
        print(f"\n  Random effects:")
        print(f"    Subject variance : {fit.cov_re.values[0, 0]:.4f}")
        print(f"    Residual variance: {fit.scale:.4f}")
    else:
        try:
            print(f"\n  R²: {fit.rsquared:.4f}  |  Adj R²: {fit.rsquared_adj:.4f}")
        except AttributeError:
            pass

    print()
    return summary_rows


# ============================================================================
# RUN MODELS
# ============================================================================

formula_osc  = 'clustering_mean ~ theta_osc_mean  * clustering_type * region'
formula_frac = 'clustering_mean ~ theta_frac_mean * clustering_type * region'

rows_osc  = fit_model(long_df, formula_osc,  param_labels_osc,  'OSC  (oscillatory)')
rows_frac = fit_model(long_df, formula_frac, param_labels_frac, 'FRAC (aperiodic)')

# ============================================================================
# BIVARIATE CORRELATIONS  (per clustering type × region)
# ============================================================================

print("=" * 80)
print("BIVARIATE CORRELATIONS  (subject-level means)")
print("=" * 80)
print(f"\n  {'Component':<8} {'Region':<6} {'Clust type':<10} "
      f"{'r':>7} {'p':>8}  {'n':>4}")
print("  " + "─" * 48)

corr_rows = []
for comp, gcol in [('osc', 'theta_osc_mean'), ('frac', 'theta_frac_mean')]:
    for region in ['RHP', 'LHP']:
        for ctype, ccol in [('spatial', 'SP_clust_mean'), ('temporal', 'T_clust_mean')]:
            sub = subj_df[subj_df['region'] == region].dropna(subset=[gcol, ccol])
            if len(sub) < 3:
                continue
            r, p = stats.pearsonr(sub[gcol], sub[ccol])
            sig  = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else ''))
            print(f"  {comp:<8} {region:<6} {ctype:<10} {r:>7.3f} {p:>8.4f}  {len(sub):>4}  {sig}")
            corr_rows.append(dict(component=comp, region=region,
                                  clust_type=ctype, r=r, p=p, n=len(sub)))
print()

# ============================================================================
# PLOTTING
# ============================================================================

colors_osc  = {'main': '#1a9641', '2-way': '#a6d96a', '3-way': '#ffffbf'}
colors_frac = {'main': '#762a83', '2-way': '#c2a5cf', '3-way': '#f7f7f7'}

def effect_color(label, cdict):
    n = label.count('×')
    return cdict['3-way'] if n == 2 else cdict['2-way'] if n == 1 else cdict['main']


def forest_plot(summary_rows, colors, title, fname):
    df_p    = pd.DataFrame(summary_rows)
    df_p    = df_p[df_p['parameter'] != 'Intercept'].copy()
    col_map = [effect_color(r.parameter, colors) for r in df_p.itertuples()]

    fig, ax = plt.subplots(figsize=(10, 5))
    fig.suptitle(title, fontsize=12, fontweight='bold')

    ypos = np.arange(len(df_p))
    ax.barh(ypos, df_p['coef'], xerr=df_p['se'],
            color=col_map, alpha=0.85, height=0.6, edgecolor='white',
            error_kw=dict(ecolor='black', capsize=4, linewidth=1.2))

    for i, row in enumerate(df_p.itertuples()):
        if row.sig:
            ax.text(row.coef + row.se + 0.001, i, row.sig,
                    va='center', fontsize=10, fontweight='bold')

    ax.axvline(0, color='black', linewidth=0.9)
    ax.set_yticks(ypos)
    ax.set_yticklabels(df_p['parameter'], fontsize=9)
    ax.set_xlabel('Coefficient (± SE)', fontsize=11)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    patches = [
        mpatches.Patch(color=colors['main'],  label='Main effect'),
        mpatches.Patch(color=colors['2-way'], label='2-way interaction'),
        mpatches.Patch(color=colors['3-way'], label='3-way interaction'),
    ]
    ax.legend(handles=patches, fontsize=9, loc='lower right')
    plt.tight_layout()
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()


def scatter_grid(subj_df, gcol, comp_label, colors, fname):
    """2×2 scatter: region × clustering type"""
    fig, axes = plt.subplots(2, 2, figsize=(10, 8))
    fig.suptitle(f'Between-Subject: Theta {comp_label} vs Clustering\n'
                 f'(subject-level means, {THETA_LO}–{THETA_HI} Hz)',
                 fontsize=12, fontweight='bold')

    for col_i, region in enumerate(['RHP', 'LHP']):
        for row_i, (ctype, ccol) in enumerate([('Spatial', 'SP_clust_mean'),
                                               ('Temporal', 'T_clust_mean')]):
            ax  = axes[row_i, col_i]
            sub = subj_df[subj_df['region'] == region].dropna(subset=[gcol, ccol])

            ax.scatter(sub[gcol], sub[ccol],
                       color=colors['main'], alpha=0.75, s=60, edgecolors='white')

            if len(sub) >= 3:
                m, b, r, p, _ = stats.linregress(sub[gcol], sub[ccol])
                xline = np.linspace(sub[gcol].min(), sub[gcol].max(), 100)
                ls    = '-' if p < 0.05 else '--'
                ax.plot(xline, m * xline + b, color='black', linewidth=1.5,
                        linestyle=ls)
                sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
                ax.set_title(f'{region} | {ctype}\nr = {r:.2f}, {sig}', fontsize=10)
            else:
                ax.set_title(f'{region} | {ctype}\nn too small', fontsize=10)

            ax.set_xlabel(f'Theta {comp_label} (mean)', fontsize=9)
            ax.set_ylabel(f'{ctype} clustering (mean)', fontsize=9)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()


# Forest plots
forest_plot(rows_osc,
            colors_osc,
            f'Between-Subject: Theta Oscillatory → Clustering  ({THETA_LO}–{THETA_HI} Hz)\n'
            f'LMM Coefficients ± SE',
            'forest_between_theta_osc.png')

forest_plot(rows_frac,
            colors_frac,
            f'Between-Subject: Theta Fractal (Aperiodic) → Clustering  ({THETA_LO}–{THETA_HI} Hz)\n'
            f'LMM Coefficients ± SE',
            'forest_between_theta_frac.png')

# Scatter grids
scatter_grid(subj_df, 'theta_osc_mean',  'Oscillatory', colors_osc,
             'scatter_between_theta_osc.png')
scatter_grid(subj_df, 'theta_frac_mean', 'Fractal',     colors_frac,
             'scatter_between_theta_frac.png')

print("=" * 60)
print("DONE")
print("=" * 60)

Loaded 27,162 rows | 26 subjects
Theta-band rows (8–16 Hz): 4,527
Trial-level theta rows: 1,509 | Subjects: 26

Subject-level rows (subject × region): 38
  Subjects: 26
  Trials per subject (mean ± SD): 39.7 ± 22.9

subject region  n_trials  theta_osc_mean  theta_frac_mean  SP_clust_mean  T_clust_mean
 R1494D    RHP        24        0.192681         2.987559       0.593954      0.645098
 R1501J    RHP        69        0.010905         3.358240       0.519660      0.539609
 R1501J    LHP        69       -0.055673         3.313736       0.519660      0.539609
 R1502D    RHP        57       -0.154362         2.300974       0.491887      0.632734
 R1502D    LHP        57       -0.204492         2.185252       0.491887      0.632734
 R1503E    LHP        83        0.180534         0.990982       0.520871      0.576596
 R1504E    RHP        19       -0.118992         1.199443       0.526058      0.663267
 R1509E    LHP       109        0.237080         1.509062       0.597070      0.599987
 

In [11]:
"""
Theta-band iEEG LMM Analysis
=============================
Identical model structure to the gamma analysis, applied to theta (2–5 Hz).

Model structure
---------------
BETWEEN-SUBJECT (Level-2 OLS):
  clustering_score_j ~ neural_between_j + clust_T + region_RHP
                     + neural_between_j:clust_T
                     + neural_between_j:region_RHP
  Data: aggregated to subject × clust_T × region means (averaged over freq_hz)

WITHIN-SUBJECT (LMM):
  clustering_score_ij ~ neural_within_ij + clust_T + region_RHP + freq_hz
                      + neural_within_ij:clust_T
                      + neural_within_ij:region_RHP
                      + neural_within_ij:freq_hz
                      + (1 + neural_within | subject)
                      + (1 | subject:session)
  Optimizer: powell
"""

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# LOAD & FILTER TO THETA (2–5 Hz)
# ============================================================================

df = pd.read_csv('ALL_SUBJECTS_irasa_clean.csv')

theta = df[(df['freq_hz'] >= 2) & (df['freq_hz'] <= 12)].copy()
theta = theta[theta['SP_clustering_from_prev'].notna()].copy()

print(f"Theta band rows: {len(theta)}")
print(f"Unique freq_hz values in theta: {sorted(theta['freq_hz'].unique())}")
print(f"Unique subjects: {theta['subject'].nunique()}")

# ============================================================================
# RESHAPE: clustering_type as factor
# ============================================================================

sp = theta.copy()
sp['clustering_score'] = sp['SP_clustering_from_prev']
sp['clustering_type']  = 'SP'

t = theta.copy()
t['clustering_score'] = t['T_clustering_from_prev']
t['clustering_type']  = 'T'

long = pd.concat([sp, t], ignore_index=True)
long['clust_T']    = (long['clustering_type'] == 'T').astype(int)
long['region_RHP'] = (long['region'] == 'RHP').astype(int)
long['sess_id']    = long['subject'].astype(str) + '_' + long['session'].astype(str)

# ============================================================================
# BETWEEN / WITHIN DECOMPOSITION
# ============================================================================

for col in ['encoding_osc_ssl', 'encoding_frac_ssl',
            'retrieval_osc_ssl', 'retrieval_frac_ssl']:
    subj_mean              = long.groupby('subject')[col].transform('mean')
    long[col + '_between'] = subj_mean
    long[col + '_within']  = long[col] - subj_mean

# ============================================================================
# EFFECT SIZE HELPERS
# ============================================================================

def partial_eta_sq(t_val, df_resid):
    """Partial eta-squared from t-statistic and residual df."""
    if pd.isna(t_val) or pd.isna(df_resid):
        return np.nan
    return t_val**2 / (t_val**2 + df_resid)

def cohens_d_from_t(t_val, n):
    """Approximate Cohen's d from t and sample size (one-sample style)."""
    if pd.isna(t_val) or pd.isna(n) or n == 0:
        return np.nan
    return t_val / np.sqrt(n)

# ============================================================================
# MODELS
# ============================================================================

PHASES = ['encoding', 'retrieval']
COMPS  = ['osc', 'frac']

results_rows = []

for phase in PHASES:
    for comp in COMPS:

        col_base    = f'{phase}_{comp}_ssl'
        col_between = col_base + '_between'
        col_within  = col_base + '_within'

        # ── BETWEEN-SUBJECT MODEL ─────────────────────────────────────────
        # Aggregate to subject × clust_T × region ONLY (no freq_hz)
        between_data = (
            long.groupby(['subject', 'clust_T', 'region_RHP'])
            [[col_between, 'clustering_score']]
            .mean()
            .reset_index()
            .dropna()
        )

        try:
            formula_b = (
                f'clustering_score ~ {col_between} + clust_T + region_RHP '
                f'+ {col_between}:clust_T '
                f'+ {col_between}:region_RHP'
            )
            res_b      = smf.ols(formula_b, data=between_data).fit()
            df_resid_b = res_b.df_resid
            n_subj     = between_data['subject'].nunique()

            def gb(key):
                t_val = res_b.tvalues.get(key, np.nan)
                p_val = res_b.pvalues.get(key, np.nan)
                e_val = partial_eta_sq(t_val, df_resid_b)
                return t_val, p_val, e_val

            t_b,         p_b,         e_b         = gb(col_between)
            t_b_xclust,  p_b_xclust,  e_b_xclust  = gb(f'{col_between}:clust_T')
            t_b_xregion, p_b_xregion, e_b_xregion = gb(f'{col_between}:region_RHP')
            t_clust_b,   p_clust_b,   e_clust_b   = gb('clust_T')
            t_region_b,  p_region_b,  e_region_b  = gb('region_RHP')
            d_b    = cohens_d_from_t(t_b, n_subj)
            n_b    = int(res_b.nobs)
            note_b = 'ok'

        except Exception as exc:
            (t_b, p_b, e_b, t_b_xclust, p_b_xclust, e_b_xclust,
             t_b_xregion, p_b_xregion, e_b_xregion,
             t_clust_b, p_clust_b, e_clust_b,
             t_region_b, p_region_b, e_region_b,
             d_b, n_b, n_subj) = [np.nan] * 18
            note_b = f'failed: {exc}'

        # ── WITHIN-SUBJECT MODEL (LMM) ────────────────────────────────────
        within_data = long[[
            'subject', 'sess_id', 'clustering_score',
            col_within, 'clust_T', 'region_RHP', 'freq_hz'
        ]].dropna().copy()
        within_data = within_data.rename(columns={col_within: 'neural_within'})

        try:
            formula_w = (
                'clustering_score ~ neural_within + clust_T + region_RHP + freq_hz '
                '+ neural_within:clust_T '
                '+ neural_within:region_RHP '
                '+ neural_within:freq_hz'
            )
            md = smf.mixedlm(
                formula_w,
                data=within_data,
                groups=within_data['subject'],
                re_formula='~neural_within',
                vc_formula={'sess_id': '0 + C(sess_id)'}
            )
            res_w      = md.fit(reml=True, method='powell')
            df_resid_w = res_w.df_resid

            def gw(key):
                t_val = res_w.tvalues.get(key, np.nan)
                p_val = res_w.pvalues.get(key, np.nan)
                e_val = partial_eta_sq(t_val, df_resid_w)
                return t_val, p_val, e_val

            t_w,         p_w,         e_w         = gw('neural_within')
            t_w_xclust,  p_w_xclust,  e_w_xclust  = gw('neural_within:clust_T')
            t_w_xregion, p_w_xregion, e_w_xregion = gw('neural_within:region_RHP')
            t_w_xfreq,   p_w_xfreq,   e_w_xfreq   = gw('neural_within:freq_hz')
            t_clust_w,   p_clust_w,   e_clust_w   = gw('clust_T')
            t_region_w,  p_region_w,  e_region_w  = gw('region_RHP')
            t_freq_w,    p_freq_w,    e_freq_w    = gw('freq_hz')
            d_w    = cohens_d_from_t(t_w, within_data['subject'].nunique())
            n_w    = int(res_w.nobs)
            conv_w = res_w.converged
            note_w = 'ok' if conv_w else 'ok (convergence warning)'

        except Exception as exc:
            (t_w, p_w, e_w, t_w_xclust, p_w_xclust, e_w_xclust,
             t_w_xregion, p_w_xregion, e_w_xregion,
             t_w_xfreq, p_w_xfreq, e_w_xfreq,
             t_clust_w, p_clust_w, e_clust_w,
             t_region_w, p_region_w, e_region_w,
             t_freq_w, p_freq_w, e_freq_w,
             d_w, n_w) = [np.nan] * 23
            note_w = f'failed: {exc}'

        results_rows.append({
            'phase': phase, 'component': comp,
            # Between
            't_between':          t_b,         'p_between':          p_b,         'eta2p_between':          e_b,
            't_between_x_clustT': t_b_xclust,  'p_between_x_clustT': p_b_xclust,  'eta2p_between_x_clustT': e_b_xclust,
            't_between_x_RHP':    t_b_xregion, 'p_between_x_RHP':    p_b_xregion, 'eta2p_between_x_RHP':    e_b_xregion,
            't_clustT_between':   t_clust_b,   'p_clustT_between':   p_clust_b,   'eta2p_clustT_between':   e_clust_b,
            't_region_between':   t_region_b,  'p_region_between':   p_region_b,  'eta2p_region_between':   e_region_b,
            'd_between':          d_b,
            'n_between':          n_b,         'n_subjects':         n_subj,      'note_between': note_b,
            # Within
            't_within':           t_w,         'p_within':           p_w,         'eta2p_within':           e_w,
            't_within_x_clustT':  t_w_xclust,  'p_within_x_clustT':  p_w_xclust,  'eta2p_within_x_clustT':  e_w_xclust,
            't_within_x_RHP':     t_w_xregion, 'p_within_x_RHP':     p_w_xregion, 'eta2p_within_x_RHP':     e_w_xregion,
            't_within_x_freq':    t_w_xfreq,   'p_within_x_freq':    p_w_xfreq,   'eta2p_within_x_freq':    e_w_xfreq,
            't_clustT_within':    t_clust_w,   'p_clustT_within':    p_clust_w,   'eta2p_clustT_within':    e_clust_w,
            't_region_within':    t_region_w,  'p_region_within':    p_region_w,  'eta2p_region_within':    e_region_w,
            't_freq_within':      t_freq_w,    'p_freq_within':      p_freq_w,    'eta2p_freq_within':      e_freq_w,
            'd_within':           d_w,
            'n_within':           n_w,         'note_within':        note_w,
        })

results = pd.DataFrame(results_rows)

# ============================================================================
# FDR CORRECTION (Benjamini-Hochberg)
# Applied separately within between-subject and within-subject levels
# ============================================================================

def apply_fdr(results, p_cols):
    all_p = []
    for pc in p_cols:
        all_p.extend(results[pc].tolist())
    all_p     = np.array(all_p, dtype=float)
    valid_mask = ~np.isnan(all_p)
    fdr_p      = np.full_like(all_p, np.nan)
    if valid_mask.sum() > 0:
        _, p_corrected, _, _ = multipletests(all_p[valid_mask], method='fdr_bh')
        fdr_p[valid_mask] = p_corrected
    idx = 0
    for pc in p_cols:
        n       = len(results)
        col_fdr = pc.replace('p_', 'pfdr_')
        col_sig = pc.replace('p_', 'sig_fdr_')
        results[col_fdr] = fdr_p[idx:idx+n]
        results[col_sig] = results[col_fdr].apply(
            lambda x: x < 0.05 if pd.notna(x) else False)
        idx += n
    return results

between_p_cols = ['p_between', 'p_between_x_clustT', 'p_between_x_RHP',
                  'p_clustT_between', 'p_region_between']
within_p_cols  = ['p_within', 'p_within_x_clustT', 'p_within_x_RHP',
                  'p_within_x_freq', 'p_clustT_within', 'p_region_within', 'p_freq_within']

results = apply_fdr(results, between_p_cols)
results = apply_fdr(results, within_p_cols)

for pc in between_p_cols + within_p_cols:
    results['sig_unc_' + pc[2:]] = results[pc].apply(
        lambda x: x < 0.05 if pd.notna(x) else False)

# ============================================================================
# SAVE RESULTS
# ============================================================================

results.to_csv('theta_lmm_results.csv', index=False)

# ============================================================================
# VISUALIZATION
# Key interaction: Neural × clustering_type (OSC vs FRAC, encoding vs retrieval)
# 2×2 panel: rows = phase, cols = component
# ============================================================================

def get_between_plot_data(long, col_between):
    data = (
        long.groupby(['subject', 'clust_T'])
        [[col_between, 'clustering_score']]
        .mean()
        .reset_index()
        .dropna()
    )
    sp_data = data[data['clust_T'] == 0].copy()
    t_data  = data[data['clust_T'] == 1].copy()
    return sp_data, t_data

COL_SP   = '#2166AC'
COL_T    = '#D6604D'
COL_GRID = '#E8E8E8'

fig = plt.figure(figsize=(12, 10))
fig.patch.set_facecolor('white')
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.38)

panel_labels = [['A', 'B'], ['C', 'D']]

for pi, phase in enumerate(PHASES):
    for ci, comp in enumerate(COMPS):

        ax          = fig.add_subplot(gs[pi, ci])
        col_base    = f'{phase}_{comp}_ssl'
        col_between = col_base + '_between'

        sp_data, t_data = get_between_plot_data(long, col_between)

        ax.scatter(sp_data[col_between], sp_data['clustering_score'],
                   color=COL_SP, s=60, zorder=3, alpha=0.85,
                   label='SP clustering', edgecolors='white', linewidths=0.5)
        ax.scatter(t_data[col_between],  t_data['clustering_score'],
                   color=COL_T,  s=60, zorder=3, alpha=0.85,
                   label='T clustering',  edgecolors='white', linewidths=0.5)

        for dat, col in [(sp_data, COL_SP), (t_data, COL_T)]:
            x = dat[col_between].values
            y = dat['clustering_score'].values
            if len(x) > 1:
                m, b = np.polyfit(x, y, 1)
                xr   = np.linspace(x.min(), x.max(), 100)
                ax.plot(xr, m*xr + b, color=col, linewidth=2, alpha=0.9)

        row    = results[(results['phase']==phase) & (results['component']==comp)].iloc[0]
        t_main = row['t_between'];          p_main = row['pfdr_between']
        t_int  = row['t_between_x_clustT']; p_int  = row['pfdr_between_x_clustT']
        d_main = row['d_between']

        def pstr(p):
            if pd.isna(p):  return 'n.s.'
            if p < 0.001:   return 'p<.001'
            if p < 0.05:    return f'p={p:.3f}'
            return f'p={p:.2f} n.s.'

        ann = (f"Neural main: t={t_main:+.2f}, {pstr(p_main)}, d={d_main:.2f}\n"
               f"× Clust type: t={t_int:+.2f}, {pstr(p_int)}")
        ax.text(0.03, 0.97, ann, transform=ax.transAxes,
                fontsize=7.5, va='top', ha='left',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='#F7F7F7',
                          edgecolor='#CCCCCC', alpha=0.9))

        ax.text(-0.12, 1.05, panel_labels[pi][ci], transform=ax.transAxes,
                fontsize=14, fontweight='bold', va='top')

        comp_label  = 'Oscillatory' if comp == 'osc' else 'Aperiodic (Fractal)'
        phase_label = phase.capitalize()
        ax.set_title(f'{phase_label} — {comp_label} θ', fontsize=11, fontweight='bold', pad=8)
        ax.set_xlabel('Theta power (subject mean, SSL)', fontsize=9)
        ax.set_ylabel('Clustering score', fontsize=9)

        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.grid(True, color=COL_GRID, linewidth=0.7, zorder=0)
        ax.tick_params(labelsize=8)

        if pi == 0 and ci == 0:
            ax.legend(fontsize=8.5, framealpha=0.9, loc='lower right')

fig.suptitle(
    'Between-subject theta power (2–5 Hz) predicts semantic clustering\n'
    'across encoding and retrieval phases (iEEG)',
    fontsize=13, fontweight='bold', y=1.01
)

plt.savefig('theta_neural_x_clustering_type.png',
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("Figure saved: theta_neural_x_clustering_type.png")

# ============================================================================
# PRINT REPORT
# ============================================================================

lines = []
L = lines.append

L("=" * 110)
L("THETA-BAND (2–5 Hz) iEEG LMM — Between- and Within-Subject Effects on Clustering")
L("  Frequency range : 2–5 Hz (theta)")
L("  FDR correction  : Benjamini-Hochberg, applied within each model level")
L("  Effect sizes    : partial η²p and Cohen's d")
L("  Factors         : clustering_type (SP=0/T=1), region (LHP=0/RHP=1)")
L("  Sig             : * uncorrected p<.05  | † FDR-corrected p<.05")
L("=" * 110)

def fmt(label, t, p_unc, p_fdr, eta2, d=None):
    sig_unc = '*'  if (pd.notna(p_unc) and p_unc < 0.05) else ''
    sig_fdr = '†' if (pd.notna(p_fdr) and p_fdr < 0.05) else ''
    ts  = f'{t:+.3f}'    if pd.notna(t)     else '    NaN'
    pu  = f'{p_unc:.3f}' if pd.notna(p_unc) else '    NaN'
    pf  = f'{p_fdr:.3f}' if pd.notna(p_fdr) else '    NaN'
    e2  = f'{eta2:.3f}'  if pd.notna(eta2)  else '    NaN'
    d_s = f'{d:+.3f}'    if (d is not None and pd.notna(d)) else '      —'
    return (f"  {label:<40} {ts:>9} {pu:>9} {pf:>9} {e2:>8} {d_s:>9}  {sig_unc}{sig_fdr}")

HDR = f"  {'Effect':<40} {'t':>9} {'p_unc':>9} {'p_fdr':>9} {'η²p':>8} {'d':>9}  sig"
SEP = f"  {'─' * 95}"

for _, row in results.iterrows():
    phase = row['phase']; comp = row['component']
    L(f"\n{'━' * 110}")
    L(f"  PHASE: {phase.upper()}   COMPONENT: {'OSCILLATORY (OSC)' if comp=='osc' else 'APERIODIC/FRACTAL (FRAC)'}")
    L(f"{'━' * 110}")

    if row['note_between'] == 'ok':
        L(f"\n  BETWEEN-SUBJECT (Level-2 OLS)  "
          f"n_subjects={int(row['n_subjects'])}  n_obs={int(row['n_between'])}")
        L(f"  * uncorrected p<.05  † FDR-corrected p<.05")
        L(HDR); L(SEP)
        L(fmt('Neural main effect',        row['t_between'],          row['p_between'],          row['pfdr_between'],          row['eta2p_between'],         row['d_between']))
        L(fmt('Clustering type (T vs SP)', row['t_clustT_between'],   row['p_clustT_between'],   row['pfdr_clustT_between'],   row['eta2p_clustT_between']))
        L(fmt('Region (RHP vs LHP)',       row['t_region_between'],   row['p_region_between'],   row['pfdr_region_between'],   row['eta2p_region_between']))
        L(fmt('Neural × clustering_type', row['t_between_x_clustT'], row['p_between_x_clustT'], row['pfdr_between_x_clustT'], row['eta2p_between_x_clustT']))
        L(fmt('Neural × region',          row['t_between_x_RHP'],    row['p_between_x_RHP'],    row['pfdr_between_x_RHP'],    row['eta2p_between_x_RHP']))
    else:
        L(f"\n  BETWEEN-SUBJECT  *** {row['note_between']} ***")

    status = row['note_within']
    if status.startswith('ok'):
        L(f"\n  WITHIN-SUBJECT (LMM: random slope + session nesting)  "
          f"n_obs={int(row['n_within'])}  [{status}]")
        L(f"  * uncorrected p<.05  † FDR-corrected p<.05")
        L(HDR); L(SEP)
        L(fmt('Neural main effect',        row['t_within'],           row['p_within'],           row['pfdr_within'],           row['eta2p_within'],          row['d_within']))
        L(fmt('Clustering type (T vs SP)', row['t_clustT_within'],    row['p_clustT_within'],    row['pfdr_clustT_within'],    row['eta2p_clustT_within']))
        L(fmt('Region (RHP vs LHP)',       row['t_region_within'],    row['p_region_within'],    row['pfdr_region_within'],    row['eta2p_region_within']))
        L(fmt('Frequency (freq_hz)',       row['t_freq_within'],      row['p_freq_within'],      row['pfdr_freq_within'],      row['eta2p_freq_within']))
        L(fmt('Neural × clustering_type', row['t_within_x_clustT'],  row['p_within_x_clustT'],  row['pfdr_within_x_clustT'],  row['eta2p_within_x_clustT']))
        L(fmt('Neural × region',          row['t_within_x_RHP'],     row['p_within_x_RHP'],     row['pfdr_within_x_RHP'],     row['eta2p_within_x_RHP']))
        L(fmt('Neural × freq_hz',         row['t_within_x_freq'],    row['p_within_x_freq'],    row['pfdr_within_x_freq'],    row['eta2p_within_x_freq']))
    else:
        L(f"\n  WITHIN-SUBJECT  *** {status} ***")

L(f"\n\n{'=' * 110}")
L("FDR CORRECTION SUMMARY")
L(f"{'=' * 110}")
L("Between-subject — effects surviving FDR correction:")
for _, row in results.iterrows():
    for pc, label in [('pfdr_between',          'Neural main'),
                      ('pfdr_between_x_clustT', 'Neural × clust_T'),
                      ('pfdr_between_x_RHP',    'Neural × region'),
                      ('pfdr_clustT_between',   'Clust type main'),
                      ('pfdr_region_between',   'Region main')]:
        p = row.get(pc, np.nan)
        if pd.notna(p) and p < 0.05:
            L(f"  [{row['phase'].upper()} | {'OSC' if row['component']=='osc' else 'FRAC'}]  "
              f"{label:<30} p_fdr={p:.3f}")

L("\nWithin-subject — effects surviving FDR correction:")
found_within = False
for _, row in results.iterrows():
    for pc, label in [('pfdr_within',          'Neural main'),
                      ('pfdr_within_x_clustT', 'Neural × clust_T'),
                      ('pfdr_within_x_RHP',    'Neural × region'),
                      ('pfdr_within_x_freq',   'Neural × freq_hz'),
                      ('pfdr_clustT_within',   'Clust type main'),
                      ('pfdr_region_within',   'Region main'),
                      ('pfdr_freq_within',     'Freq main')]:
        p = row.get(pc, np.nan)
        if pd.notna(p) and p < 0.05:
            found_within = True
            L(f"  [{row['phase'].upper()} | {'OSC' if row['component']=='osc' else 'FRAC'}]  "
              f"{label:<30} p_fdr={p:.3f}")
if not found_within:
    L("  None.")

L(f"\n{'=' * 110}")
L("FILES SAVED:")
L("  theta_lmm_results.csv              — full results with FDR p-values and effect sizes")
L("  theta_lmm_summary.csv              — FDR-significant effects only")
L("  theta_neural_x_clustering_type.png — Figure: Neural × clustering_type (2×2 panel)")
L("=" * 110)

report = '\n'.join(lines)
print(report)

with open('theta_lmm_report.txt', 'w') as f:
    f.write(report)

sig_fdr_cols = [c for c in results.columns if c.startswith('sig_fdr_')]
results[results[sig_fdr_cols].any(axis=1)].to_csv('theta_lmm_summary.csv', index=False)

print("\nDone.")

Theta band rows: 8477
Unique freq_hz values in theta: [3.0, 3.74, 4.67, 5.82, 7.26, 9.05, 11.28]
Unique subjects: 26
Figure saved: theta_neural_x_clustering_type.png
THETA-BAND (2–5 Hz) iEEG LMM — Between- and Within-Subject Effects on Clustering
  Frequency range : 2–5 Hz (theta)
  FDR correction  : Benjamini-Hochberg, applied within each model level
  Effect sizes    : partial η²p and Cohen's d
  Factors         : clustering_type (SP=0/T=1), region (LHP=0/RHP=1)
  Sig             : * uncorrected p<.05  | † FDR-corrected p<.05

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  PHASE: ENCODING   COMPONENT: OSCILLATORY (OSC)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  BETWEEN-SUBJECT (Level-2 OLS)  n_subjects=26  n_obs=76
  * uncorrected p<.05  † FDR-corrected p<.05
  Effect                                           t     p_unc     p_fdr      η²p         d

In [13]:
"""
Gamma-band iEEG LMM Analysis — REC/LEC Version
=============================================
Changes from original:
  - Region factor: LEC (=0) vs REC (=1) instead of LHP (=0) vs RHP (=1)
  - Filter: only rows where region in ['LEC', 'REC']
  - All variable names, labels, and column references updated accordingly
"""

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# LOAD & FILTER TO GAMMA + LEC/REC ONLY
# ============================================================================

df = pd.read_csv('ALL_SUBJECTS_irasa_per_freq-Copy1.csv')

gamma = df[df['freq_hz'] >= 40].copy()
gamma = gamma[gamma['region'].isin(['LEC', 'REC'])].copy()
gamma = gamma[gamma['SP_clustering_from_prev'].notna()].copy()

# ============================================================================
# RESHAPE: clustering_type as factor
# ============================================================================

sp = gamma.copy()
sp['clustering_score'] = sp['SP_clustering_from_prev']
sp['clustering_type']  = 'SP'

t = gamma.copy()
t['clustering_score'] = t['T_clustering_from_prev']
t['clustering_type']  = 'T'

long = pd.concat([sp, t], ignore_index=True)
long['clust_T']    = (long['clustering_type'] == 'T').astype(int)
long['region_REC'] = (long['region'] == 'REC').astype(int)   # LEC=0, REC=1
long['sess_id']    = long['subject'].astype(str) + '_' + long['session'].astype(str)

# ============================================================================
# BETWEEN / WITHIN DECOMPOSITION
# ============================================================================

for col in ['encoding_osc_ssl', 'encoding_frac_ssl',
            'retrieval_osc_ssl', 'retrieval_frac_ssl']:
    subj_mean              = long.groupby('subject')[col].transform('mean')
    long[col + '_between'] = subj_mean
    long[col + '_within']  = long[col] - subj_mean

# ============================================================================
# EFFECT SIZE HELPERS
# ============================================================================

def partial_eta_sq(t_val, df_resid):
    if pd.isna(t_val) or pd.isna(df_resid):
        return np.nan
    return t_val**2 / (t_val**2 + df_resid)

def cohens_d_from_t(t_val, n):
    if pd.isna(t_val) or pd.isna(n) or n == 0:
        return np.nan
    return t_val / np.sqrt(n)

# ============================================================================
# MODELS
# ============================================================================

PHASES = ['encoding', 'retrieval']
COMPS  = ['osc', 'frac']

results_rows = []

for phase in PHASES:
    for comp in COMPS:

        col_base    = f'{phase}_{comp}_ssl'
        col_between = col_base + '_between'
        col_within  = col_base + '_within'

        # ── BETWEEN-SUBJECT MODEL ─────────────────────────────────────────
        between_data = (
            long.groupby(['subject', 'clust_T', 'region_REC'])
            [[col_between, 'clustering_score']]
            .mean()
            .reset_index()
            .dropna()
        )

        try:
            formula_b = (
                f'clustering_score ~ {col_between} + clust_T + region_REC '
                f'+ {col_between}:clust_T '
                f'+ {col_between}:region_REC'
            )
            res_b      = smf.ols(formula_b, data=between_data).fit()
            df_resid_b = res_b.df_resid
            n_subj     = between_data['subject'].nunique()

            def gb(key):
                t = res_b.tvalues.get(key, np.nan)
                p = res_b.pvalues.get(key, np.nan)
                e = partial_eta_sq(t, df_resid_b)
                return t, p, e

            t_b,         p_b,         e_b         = gb(col_between)
            t_b_xclust,  p_b_xclust,  e_b_xclust  = gb(f'{col_between}:clust_T')
            t_b_xregion, p_b_xregion, e_b_xregion = gb(f'{col_between}:region_REC')
            t_clust_b,   p_clust_b,   e_clust_b   = gb('clust_T')
            t_region_b,  p_region_b,  e_region_b  = gb('region_REC')
            d_b  = cohens_d_from_t(t_b, n_subj)
            n_b  = int(res_b.nobs)
            note_b = 'ok'

        except Exception as e:
            (t_b, p_b, e_b, t_b_xclust, p_b_xclust, e_b_xclust,
             t_b_xregion, p_b_xregion, e_b_xregion,
             t_clust_b, p_clust_b, e_clust_b,
             t_region_b, p_region_b, e_region_b,
             d_b, n_b, n_subj) = [np.nan] * 18
            note_b = f'failed: {e}'

        # ── WITHIN-SUBJECT MODEL (LMM) ────────────────────────────────────
        within_data = long[[
            'subject', 'sess_id', 'clustering_score',
            col_within, 'clust_T', 'region_REC', 'freq_hz'
        ]].dropna().copy()
        within_data = within_data.rename(columns={col_within: 'neural_within'})

        try:
            formula_w = (
                'clustering_score ~ neural_within + clust_T + region_REC + freq_hz '
                '+ neural_within:clust_T '
                '+ neural_within:region_REC '
                '+ neural_within:freq_hz'
            )
            md = smf.mixedlm(
                formula_w,
                data=within_data,
                groups=within_data['subject'],
                re_formula='~neural_within',
                vc_formula={'sess_id': '0 + C(sess_id)'}
            )
            res_w      = md.fit(reml=True, method='powell')
            df_resid_w = res_w.df_resid

            def gw(key):
                t = res_w.tvalues.get(key, np.nan)
                p = res_w.pvalues.get(key, np.nan)
                e = partial_eta_sq(t, df_resid_w)
                return t, p, e

            t_w,         p_w,         e_w         = gw('neural_within')
            t_w_xclust,  p_w_xclust,  e_w_xclust  = gw('neural_within:clust_T')
            t_w_xregion, p_w_xregion, e_w_xregion = gw('neural_within:region_REC')
            t_w_xfreq,   p_w_xfreq,   e_w_xfreq   = gw('neural_within:freq_hz')
            t_clust_w,   p_clust_w,   e_clust_w   = gw('clust_T')
            t_region_w,  p_region_w,  e_region_w  = gw('region_REC')
            t_freq_w,    p_freq_w,    e_freq_w    = gw('freq_hz')
            d_w    = cohens_d_from_t(t_w, within_data['subject'].nunique())
            n_w    = int(res_w.nobs)
            conv_w = res_w.converged
            note_w = 'ok' if conv_w else 'ok (convergence warning)'

        except Exception as e:
            (t_w, p_w, e_w, t_w_xclust, p_w_xclust, e_w_xclust,
             t_w_xregion, p_w_xregion, e_w_xregion,
             t_w_xfreq, p_w_xfreq, e_w_xfreq,
             t_clust_w, p_clust_w, e_clust_w,
             t_region_w, p_region_w, e_region_w,
             t_freq_w, p_freq_w, e_freq_w,
             d_w, n_w) = [np.nan] * 23
            note_w = f'failed: {e}'

        results_rows.append({
            'phase': phase, 'component': comp,
            # Between
            't_between':          t_b,         'p_between':          p_b,         'eta2p_between':          e_b,
            't_between_x_clustT': t_b_xclust,  'p_between_x_clustT': p_b_xclust,  'eta2p_between_x_clustT': e_b_xclust,
            't_between_x_REC':    t_b_xregion, 'p_between_x_REC':    p_b_xregion, 'eta2p_between_x_REC':    e_b_xregion,
            't_clustT_between':   t_clust_b,   'p_clustT_between':   p_clust_b,   'eta2p_clustT_between':   e_clust_b,
            't_region_between':   t_region_b,  'p_region_between':   p_region_b,  'eta2p_region_between':   e_region_b,
            'd_between':          d_b,
            'n_between':          n_b,         'n_subjects':         n_subj,      'note_between': note_b,
            # Within
            't_within':           t_w,         'p_within':           p_w,         'eta2p_within':           e_w,
            't_within_x_clustT':  t_w_xclust,  'p_within_x_clustT':  p_w_xclust,  'eta2p_within_x_clustT':  e_w_xclust,
            't_within_x_REC':     t_w_xregion, 'p_within_x_REC':     p_w_xregion, 'eta2p_within_x_REC':     e_w_xregion,
            't_within_x_freq':    t_w_xfreq,   'p_within_x_freq':    p_w_xfreq,   'eta2p_within_x_freq':    e_w_xfreq,
            't_clustT_within':    t_clust_w,   'p_clustT_within':    p_clust_w,   'eta2p_clustT_within':    e_clust_w,
            't_region_within':    t_region_w,  'p_region_within':    p_region_w,  'eta2p_region_within':    e_region_w,
            't_freq_within':      t_freq_w,    'p_freq_within':      p_freq_w,    'eta2p_freq_within':      e_freq_w,
            'd_within':           d_w,
            'n_within':           n_w,         'note_within':        note_w,
        })

results = pd.DataFrame(results_rows)

# ============================================================================
# FDR CORRECTION (Benjamini-Hochberg)
# ============================================================================

def apply_fdr(results, p_cols, suffix):
    all_p = []
    for pc in p_cols:
        all_p.extend(results[pc].tolist())
    all_p = np.array(all_p, dtype=float)
    valid_mask = ~np.isnan(all_p)
    fdr_p = np.full_like(all_p, np.nan)
    if valid_mask.sum() > 0:
        _, p_corrected, _, _ = multipletests(all_p[valid_mask], method='fdr_bh')
        fdr_p[valid_mask] = p_corrected
    idx = 0
    for pc in p_cols:
        n = len(results)
        col_fdr = pc.replace('p_', 'pfdr_')
        col_sig = pc.replace('p_', 'sig_fdr_')
        results[col_fdr] = fdr_p[idx:idx+n]
        results[col_sig] = results[col_fdr].apply(
            lambda x: x < 0.05 if pd.notna(x) else False)
        idx += n
    return results

between_p_cols = ['p_between', 'p_between_x_clustT', 'p_between_x_REC',
                  'p_clustT_between', 'p_region_between']
within_p_cols  = ['p_within', 'p_within_x_clustT', 'p_within_x_REC',
                  'p_within_x_freq', 'p_clustT_within', 'p_region_within', 'p_freq_within']

results = apply_fdr(results, between_p_cols, 'between')
results = apply_fdr(results, within_p_cols,  'within')

for pc in between_p_cols + within_p_cols:
    results['sig_unc_' + pc[2:]] = results[pc].apply(
        lambda x: x < 0.05 if pd.notna(x) else False)

# ============================================================================
# SAVE RESULTS
# ============================================================================

results.to_csv('gamma_lmm_results_EC.csv', index=False)

# ============================================================================
# VISUALIZATION
# ============================================================================

def get_between_plot_data(long, col_between):
    data = (
        long.groupby(['subject', 'clust_T'])
        [[col_between, 'clustering_score']]
        .mean()
        .reset_index()
        .dropna()
    )
    sp_data = data[data['clust_T'] == 0].copy()
    t_data  = data[data['clust_T'] == 1].copy()
    return sp_data, t_data

COL_SP   = '#2166AC'
COL_T    = '#D6604D'
COL_GRID = '#E8E8E8'

fig = plt.figure(figsize=(12, 10))
fig.patch.set_facecolor('white')
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.38)

panel_labels = [['A', 'B'], ['C', 'D']]

for pi, phase in enumerate(PHASES):
    for ci, comp in enumerate(COMPS):

        ax = fig.add_subplot(gs[pi, ci])
        col_base    = f'{phase}_{comp}_ssl'
        col_between = col_base + '_between'

        sp_data, t_data = get_between_plot_data(long, col_between)

        ax.scatter(sp_data[col_between], sp_data['clustering_score'],
                   color=COL_SP, s=60, zorder=3, alpha=0.85,
                   label='SP clustering', edgecolors='white', linewidths=0.5)
        ax.scatter(t_data[col_between],  t_data['clustering_score'],
                   color=COL_T,  s=60, zorder=3, alpha=0.85,
                   label='T clustering',  edgecolors='white', linewidths=0.5)

        for dat, col in [(sp_data, COL_SP), (t_data, COL_T)]:
            x = dat[col_between].values
            y = dat['clustering_score'].values
            if len(x) > 1:
                m, b = np.polyfit(x, y, 1)
                xr = np.linspace(x.min(), x.max(), 100)
                ax.plot(xr, m*xr + b, color=col, linewidth=2, alpha=0.9)

        row = results[(results['phase']==phase) & (results['component']==comp)].iloc[0]
        t_main = row['t_between'];          p_main = row['pfdr_between']
        t_int  = row['t_between_x_clustT']; p_int  = row['pfdr_between_x_clustT']
        d_main = row['d_between']

        def pstr(p):
            if pd.isna(p):  return 'n.s.'
            if p < 0.001:   return 'p<.001'
            if p < 0.05:    return f'p={p:.3f}'
            return f'p={p:.2f} n.s.'

        ann = (f"Neural main: t={t_main:+.2f}, {pstr(p_main)}, d={d_main:.2f}\n"
               f"× Clust type: t={t_int:+.2f}, {pstr(p_int)}")
        ax.text(0.03, 0.97, ann, transform=ax.transAxes,
                fontsize=7.5, va='top', ha='left',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='#F7F7F7',
                          edgecolor='#CCCCCC', alpha=0.9))

        ax.text(-0.12, 1.05, panel_labels[pi][ci], transform=ax.transAxes,
                fontsize=14, fontweight='bold', va='top')

        comp_label  = 'Oscillatory' if comp == 'osc' else 'Aperiodic (Fractal)'
        phase_label = phase.capitalize()
        ax.set_title(f'{phase_label} — {comp_label} γ', fontsize=11, fontweight='bold', pad=8)
        ax.set_xlabel('Gamma power (subject mean, SSL)', fontsize=9)
        ax.set_ylabel('Clustering score', fontsize=9)

        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.grid(True, color=COL_GRID, linewidth=0.7, zorder=0)
        ax.tick_params(labelsize=8)

        if pi == 0 and ci == 0:
            ax.legend(fontsize=8.5, framealpha=0.9, loc='lower right')

fig.suptitle(
    'Between-subject gamma power predicts clustering\n'
    'across encoding and retrieval phases — Entorhinal Cortex (LEC/REC)',
    fontsize=13, fontweight='bold', y=1.01
)

plt.savefig('gamma_neural_x_clustering_type_EC.png',
            dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print("Figure saved: gamma_neural_x_clustering_type_EC.png")

# ============================================================================
# PRINT REPORT
# ============================================================================

lines = []
L = lines.append

L("=" * 110)
L("GAMMA-BAND iEEG LMM — Between- and Within-Subject Effects on Clustering  [LEC/REC VERSION]")
L("  Region factor: LEC (=0) vs REC (=1)")
L("  Fix 1: freq_hz removed from between model (unidentified at Level-2)")
L("  Fix 2: FDR correction (Benjamini-Hochberg) applied within each model level")
L("  Fix 3: Effect sizes reported — partial η²p and Cohen's d")
L("  Fix 4: Figure saved — gamma_neural_x_clustering_type_EC.png")
L("  Factors : clustering_type (SP=0/T=1), region (LEC=0/REC=1)")
L("  Sig     : ** uncorrected p<.05  | †† FDR-corrected p<.05")
L("=" * 110)

def fmt(label, t, p_unc, p_fdr, eta2, d=None):
    sig_unc = '*'  if (pd.notna(p_unc) and p_unc < 0.05) else ''
    sig_fdr = '†' if (pd.notna(p_fdr) and p_fdr < 0.05) else ''
    ts  = f'{t:+.3f}'    if pd.notna(t)     else '    NaN'
    pu  = f'{p_unc:.3f}' if pd.notna(p_unc) else '    NaN'
    pf  = f'{p_fdr:.3f}' if pd.notna(p_fdr) else '    NaN'
    e2  = f'{eta2:.3f}'  if pd.notna(eta2)  else '    NaN'
    d_s = f'{d:+.3f}'    if (d is not None and pd.notna(d)) else '      —'
    return (f"  {label:<40} {ts:>9} {pu:>9} {pf:>9} {e2:>8} {d_s:>9}  {sig_unc}{sig_fdr}")

HDR = f"  {'Effect':<40} {'t':>9} {'p_unc':>9} {'p_fdr':>9} {'η²p':>8} {'d':>9}  sig"
SEP = f"  {'─' * 95}"

for _, row in results.iterrows():
    phase = row['phase']; comp = row['component']
    L(f"\n{'━' * 110}")
    L(f"  PHASE: {phase.upper()}   COMPONENT: {'OSCILLATORY (OSC)' if comp=='osc' else 'APERIODIC/FRACTAL (FRAC)'}")
    L(f"{'━' * 110}")

    if row['note_between'] == 'ok':
        L(f"\n  BETWEEN-SUBJECT (Level-2 OLS)  "
          f"n_subjects={int(row['n_subjects'])}  n_obs={int(row['n_between'])}")
        L(f"  * uncorrected p<.05  † FDR-corrected p<.05")
        L(HDR); L(SEP)
        L(fmt('Neural main effect',         row['t_between'],         row['p_between'],         row['pfdr_between'],         row['eta2p_between'],         row['d_between']))
        L(fmt('Clustering type (T vs SP)',   row['t_clustT_between'],  row['p_clustT_between'],  row['pfdr_clustT_between'],  row['eta2p_clustT_between']))
        L(fmt('Region (REC vs LEC)',         row['t_region_between'],  row['p_region_between'],  row['pfdr_region_between'],  row['eta2p_region_between']))
        L(fmt('Neural × clustering_type',    row['t_between_x_clustT'],row['p_between_x_clustT'],row['pfdr_between_x_clustT'],row['eta2p_between_x_clustT']))
        L(fmt('Neural × region',             row['t_between_x_REC'],   row['p_between_x_REC'],   row['pfdr_between_x_REC'],   row['eta2p_between_x_REC']))
    else:
        L(f"\n  BETWEEN-SUBJECT  *** {row['note_between']} ***")

    status = row['note_within']
    if status.startswith('ok'):
        L(f"\n  WITHIN-SUBJECT (LMM: random slope + session nesting)  "
          f"n_obs={int(row['n_within'])}  [{status}]")
        L(f"  * uncorrected p<.05  † FDR-corrected p<.05")
        L(HDR); L(SEP)
        L(fmt('Neural main effect',         row['t_within'],          row['p_within'],          row['pfdr_within'],          row['eta2p_within'],          row['d_within']))
        L(fmt('Clustering type (T vs SP)',   row['t_clustT_within'],   row['p_clustT_within'],   row['pfdr_clustT_within'],   row['eta2p_clustT_within']))
        L(fmt('Region (REC vs LEC)',         row['t_region_within'],   row['p_region_within'],   row['pfdr_region_within'],   row['eta2p_region_within']))
        L(fmt('Frequency (freq_hz)',         row['t_freq_within'],     row['p_freq_within'],     row['pfdr_freq_within'],     row['eta2p_freq_within']))
        L(fmt('Neural × clustering_type',    row['t_within_x_clustT'], row['p_within_x_clustT'], row['pfdr_within_x_clustT'], row['eta2p_within_x_clustT']))
        L(fmt('Neural × region',             row['t_within_x_REC'],    row['p_within_x_REC'],    row['pfdr_within_x_REC'],    row['eta2p_within_x_REC']))
        L(fmt('Neural × freq_hz',            row['t_within_x_freq'],   row['p_within_x_freq'],   row['pfdr_within_x_freq'],   row['eta2p_within_x_freq']))
    else:
        L(f"\n  WITHIN-SUBJECT  *** {status} ***")

L(f"\n\n{'=' * 110}")
L("FDR CORRECTION SUMMARY")
L(f"{'=' * 110}")
L("Between-subject — effects surviving FDR correction:")
for _, row in results.iterrows():
    for pc, label in [('pfdr_between',         'Neural main'),
                      ('pfdr_between_x_clustT','Neural × clust_T'),
                      ('pfdr_between_x_REC',   'Neural × region'),
                      ('pfdr_clustT_between',  'Clust type main'),
                      ('pfdr_region_between',  'Region main')]:
        p = row.get(pc, np.nan)
        if pd.notna(p) and p < 0.05:
            L(f"  [{row['phase'].upper()} | {'OSC' if row['component']=='osc' else 'FRAC'}]  "
              f"{label:<30} p_fdr={p:.3f}")

L("\nWithin-subject — effects surviving FDR correction:")
found_within = False
for _, row in results.iterrows():
    for pc, label in [('pfdr_within',          'Neural main'),
                      ('pfdr_within_x_clustT', 'Neural × clust_T'),
                      ('pfdr_within_x_REC',    'Neural × region'),
                      ('pfdr_within_x_freq',   'Neural × freq_hz'),
                      ('pfdr_clustT_within',   'Clust type main'),
                      ('pfdr_region_within',   'Region main'),
                      ('pfdr_freq_within',     'Freq main')]:
        p = row.get(pc, np.nan)
        if pd.notna(p) and p < 0.05:
            found_within = True
            L(f"  [{row['phase'].upper()} | {'OSC' if row['component']=='osc' else 'FRAC'}]  "
              f"{label:<30} p_fdr={p:.3f}")
if not found_within:
    L("  None.")

L(f"\n{'=' * 110}")
L("FILES SAVED:")
L("  gamma_lmm_results_EC.csv                  — full results with FDR p-values and effect sizes")
L("  gamma_lmm_summary_EC.csv                  — FDR-significant effects only")
L("  gamma_neural_x_clustering_type_EC.png     — Figure: Neural × clustering_type (2×2 panel)")
L("=" * 110)

report = '\n'.join(lines)
print(report)

with open('gamma_lmm_report_EC.txt', 'w') as f:
    f.write(report)

sig_fdr_cols = [c for c in results.columns if c.startswith('sig_fdr_')]
results[results[sig_fdr_cols].any(axis=1)].to_csv('gamma_lmm_summary_EC.csv', index=False)

print("\nDone.")

Figure saved: gamma_neural_x_clustering_type_EC.png
GAMMA-BAND iEEG LMM — Between- and Within-Subject Effects on Clustering  [LEC/REC VERSION]
  Region factor: LEC (=0) vs REC (=1)
  Fix 1: freq_hz removed from between model (unidentified at Level-2)
  Fix 2: FDR correction (Benjamini-Hochberg) applied within each model level
  Fix 3: Effect sizes reported — partial η²p and Cohen's d
  Fix 4: Figure saved — gamma_neural_x_clustering_type_EC.png
  Factors : clustering_type (SP=0/T=1), region (LEC=0/REC=1)
  Sig     : ** uncorrected p<.05  | †† FDR-corrected p<.05

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  PHASE: ENCODING   COMPONENT: OSCILLATORY (OSC)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  BETWEEN-SUBJECT (Level-2 OLS)  n_subjects=14  n_obs=32
  * uncorrected p<.05  † FDR-corrected p<.05
  Effect                                           t    

In [14]:
"""
Gamma-band iEEG LMM Analysis — REC/LEC Version
=============================================
Changes from original:
  - Region factor: LEC (=0) vs REC (=1) instead of LHP (=0) vs RHP (=1)
  - Filter: only rows where region in ['LEC', 'REC']
  - All variable names, labels, and column references updated accordingly
"""

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# LOAD & FILTER TO GAMMA + LEC/REC ONLY
# ============================================================================

df = pd.read_csv('ALL_SUBJECTS_irasa_per_freq-Copy1.csv')

gamma = df[df['freq_hz'] >= 40].copy()
gamma = gamma[gamma['region'].isin(['LPC', 'RPC'])].copy()
gamma = gamma[gamma['SP_clustering_from_prev'].notna()].copy()

# ============================================================================
# RESHAPE: clustering_type as factor
# ============================================================================

sp = gamma.copy()
sp['clustering_score'] = sp['SP_clustering_from_prev']
sp['clustering_type']  = 'SP'

t = gamma.copy()
t['clustering_score'] = t['T_clustering_from_prev']
t['clustering_type']  = 'T'

long = pd.concat([sp, t], ignore_index=True)
long['clust_T']    = (long['clustering_type'] == 'T').astype(int)
long['region_REC'] = (long['region'] == 'REC').astype(int)   # LEC=0, REC=1
long['sess_id']    = long['subject'].astype(str) + '_' + long['session'].astype(str)

# ============================================================================
# BETWEEN / WITHIN DECOMPOSITION
# ============================================================================

for col in ['encoding_osc_ssl', 'encoding_frac_ssl',
            'retrieval_osc_ssl', 'retrieval_frac_ssl']:
    subj_mean              = long.groupby('subject')[col].transform('mean')
    long[col + '_between'] = subj_mean
    long[col + '_within']  = long[col] - subj_mean

# ============================================================================
# EFFECT SIZE HELPERS
# ============================================================================

def partial_eta_sq(t_val, df_resid):
    if pd.isna(t_val) or pd.isna(df_resid):
        return np.nan
    return t_val**2 / (t_val**2 + df_resid)

def cohens_d_from_t(t_val, n):
    if pd.isna(t_val) or pd.isna(n) or n == 0:
        return np.nan
    return t_val / np.sqrt(n)

# ============================================================================
# MODELS
# ============================================================================

PHASES = ['encoding', 'retrieval']
COMPS  = ['osc', 'frac']

results_rows = []

for phase in PHASES:
    for comp in COMPS:

        col_base    = f'{phase}_{comp}_ssl'
        col_between = col_base + '_between'
        col_within  = col_base + '_within'

        # ── BETWEEN-SUBJECT MODEL ─────────────────────────────────────────
        between_data = (
            long.groupby(['subject', 'clust_T', 'region_REC'])
            [[col_between, 'clustering_score']]
            .mean()
            .reset_index()
            .dropna()
        )

        try:
            formula_b = (
                f'clustering_score ~ {col_between} + clust_T + region_REC '
                f'+ {col_between}:clust_T '
                f'+ {col_between}:region_REC'
            )
            res_b      = smf.ols(formula_b, data=between_data).fit()
            df_resid_b = res_b.df_resid
            n_subj     = between_data['subject'].nunique()

            def gb(key):
                t = res_b.tvalues.get(key, np.nan)
                p = res_b.pvalues.get(key, np.nan)
                e = partial_eta_sq(t, df_resid_b)
                return t, p, e

            t_b,         p_b,         e_b         = gb(col_between)
            t_b_xclust,  p_b_xclust,  e_b_xclust  = gb(f'{col_between}:clust_T')
            t_b_xregion, p_b_xregion, e_b_xregion = gb(f'{col_between}:region_REC')
            t_clust_b,   p_clust_b,   e_clust_b   = gb('clust_T')
            t_region_b,  p_region_b,  e_region_b  = gb('region_REC')
            d_b  = cohens_d_from_t(t_b, n_subj)
            n_b  = int(res_b.nobs)
            note_b = 'ok'

        except Exception as e:
            (t_b, p_b, e_b, t_b_xclust, p_b_xclust, e_b_xclust,
             t_b_xregion, p_b_xregion, e_b_xregion,
             t_clust_b, p_clust_b, e_clust_b,
             t_region_b, p_region_b, e_region_b,
             d_b, n_b, n_subj) = [np.nan] * 18
            note_b = f'failed: {e}'

        # ── WITHIN-SUBJECT MODEL (LMM) ────────────────────────────────────
        within_data = long[[
            'subject', 'sess_id', 'clustering_score',
            col_within, 'clust_T', 'region_REC', 'freq_hz'
        ]].dropna().copy()
        within_data = within_data.rename(columns={col_within: 'neural_within'})

        try:
            formula_w = (
                'clustering_score ~ neural_within + clust_T + region_REC + freq_hz '
                '+ neural_within:clust_T '
                '+ neural_within:region_REC '
                '+ neural_within:freq_hz'
            )
            md = smf.mixedlm(
                formula_w,
                data=within_data,
                groups=within_data['subject'],
                re_formula='~neural_within',
                vc_formula={'sess_id': '0 + C(sess_id)'}
            )
            res_w      = md.fit(reml=True, method='powell')
            df_resid_w = res_w.df_resid

            def gw(key):
                t = res_w.tvalues.get(key, np.nan)
                p = res_w.pvalues.get(key, np.nan)
                e = partial_eta_sq(t, df_resid_w)
                return t, p, e

            t_w,         p_w,         e_w         = gw('neural_within')
            t_w_xclust,  p_w_xclust,  e_w_xclust  = gw('neural_within:clust_T')
            t_w_xregion, p_w_xregion, e_w_xregion = gw('neural_within:region_REC')
            t_w_xfreq,   p_w_xfreq,   e_w_xfreq   = gw('neural_within:freq_hz')
            t_clust_w,   p_clust_w,   e_clust_w   = gw('clust_T')
            t_region_w,  p_region_w,  e_region_w  = gw('region_REC')
            t_freq_w,    p_freq_w,    e_freq_w    = gw('freq_hz')
            d_w    = cohens_d_from_t(t_w, within_data['subject'].nunique())
            n_w    = int(res_w.nobs)
            conv_w = res_w.converged
            note_w = 'ok' if conv_w else 'ok (convergence warning)'

        except Exception as e:
            (t_w, p_w, e_w, t_w_xclust, p_w_xclust, e_w_xclust,
             t_w_xregion, p_w_xregion, e_w_xregion,
             t_w_xfreq, p_w_xfreq, e_w_xfreq,
             t_clust_w, p_clust_w, e_clust_w,
             t_region_w, p_region_w, e_region_w,
             t_freq_w, p_freq_w, e_freq_w,
             d_w, n_w) = [np.nan] * 23
            note_w = f'failed: {e}'

        results_rows.append({
            'phase': phase, 'component': comp,
            # Between
            't_between':          t_b,         'p_between':          p_b,         'eta2p_between':          e_b,
            't_between_x_clustT': t_b_xclust,  'p_between_x_clustT': p_b_xclust,  'eta2p_between_x_clustT': e_b_xclust,
            't_between_x_REC':    t_b_xregion, 'p_between_x_REC':    p_b_xregion, 'eta2p_between_x_REC':    e_b_xregion,
            't_clustT_between':   t_clust_b,   'p_clustT_between':   p_clust_b,   'eta2p_clustT_between':   e_clust_b,
            't_region_between':   t_region_b,  'p_region_between':   p_region_b,  'eta2p_region_between':   e_region_b,
            'd_between':          d_b,
            'n_between':          n_b,         'n_subjects':         n_subj,      'note_between': note_b,
            # Within
            't_within':           t_w,         'p_within':           p_w,         'eta2p_within':           e_w,
            't_within_x_clustT':  t_w_xclust,  'p_within_x_clustT':  p_w_xclust,  'eta2p_within_x_clustT':  e_w_xclust,
            't_within_x_REC':     t_w_xregion, 'p_within_x_REC':     p_w_xregion, 'eta2p_within_x_REC':     e_w_xregion,
            't_within_x_freq':    t_w_xfreq,   'p_within_x_freq':    p_w_xfreq,   'eta2p_within_x_freq':    e_w_xfreq,
            't_clustT_within':    t_clust_w,   'p_clustT_within':    p_clust_w,   'eta2p_clustT_within':    e_clust_w,
            't_region_within':    t_region_w,  'p_region_within':    p_region_w,  'eta2p_region_within':    e_region_w,
            't_freq_within':      t_freq_w,    'p_freq_within':      p_freq_w,    'eta2p_freq_within':      e_freq_w,
            'd_within':           d_w,
            'n_within':           n_w,         'note_within':        note_w,
        })

results = pd.DataFrame(results_rows)

# ============================================================================
# FDR CORRECTION (Benjamini-Hochberg)
# ============================================================================

def apply_fdr(results, p_cols, suffix):
    all_p = []
    for pc in p_cols:
        all_p.extend(results[pc].tolist())
    all_p = np.array(all_p, dtype=float)
    valid_mask = ~np.isnan(all_p)
    fdr_p = np.full_like(all_p, np.nan)
    if valid_mask.sum() > 0:
        _, p_corrected, _, _ = multipletests(all_p[valid_mask], method='fdr_bh')
        fdr_p[valid_mask] = p_corrected
    idx = 0
    for pc in p_cols:
        n = len(results)
        col_fdr = pc.replace('p_', 'pfdr_')
        col_sig = pc.replace('p_', 'sig_fdr_')
        results[col_fdr] = fdr_p[idx:idx+n]
        results[col_sig] = results[col_fdr].apply(
            lambda x: x < 0.05 if pd.notna(x) else False)
        idx += n
    return results

between_p_cols = ['p_between', 'p_between_x_clustT', 'p_between_x_REC',
                  'p_clustT_between', 'p_region_between']
within_p_cols  = ['p_within', 'p_within_x_clustT', 'p_within_x_REC',
                  'p_within_x_freq', 'p_clustT_within', 'p_region_within', 'p_freq_within']

results = apply_fdr(results, between_p_cols, 'between')
results = apply_fdr(results, within_p_cols,  'within')

for pc in between_p_cols + within_p_cols:
    results['sig_unc_' + pc[2:]] = results[pc].apply(
        lambda x: x < 0.05 if pd.notna(x) else False)

# ============================================================================
# SAVE RESULTS
# ============================================================================

results.to_csv('gamma_lmm_results_EC.csv', index=False)

# ============================================================================
# VISUALIZATION
# ============================================================================

def get_between_plot_data(long, col_between):
    data = (
        long.groupby(['subject', 'clust_T'])
        [[col_between, 'clustering_score']]
        .mean()
        .reset_index()
        .dropna()
    )
    sp_data = data[data['clust_T'] == 0].copy()
    t_data  = data[data['clust_T'] == 1].copy()
    return sp_data, t_data

COL_SP   = '#2166AC'
COL_T    = '#D6604D'
COL_GRID = '#E8E8E8'

fig = plt.figure(figsize=(12, 10))
fig.patch.set_facecolor('white')
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.38)

panel_labels = [['A', 'B'], ['C', 'D']]

for pi, phase in enumerate(PHASES):
    for ci, comp in enumerate(COMPS):

        ax = fig.add_subplot(gs[pi, ci])
        col_base    = f'{phase}_{comp}_ssl'
        col_between = col_base + '_between'

        sp_data, t_data = get_between_plot_data(long, col_between)

        ax.scatter(sp_data[col_between], sp_data['clustering_score'],
                   color=COL_SP, s=60, zorder=3, alpha=0.85,
                   label='SP clustering', edgecolors='white', linewidths=0.5)
        ax.scatter(t_data[col_between],  t_data['clustering_score'],
                   color=COL_T,  s=60, zorder=3, alpha=0.85,
                   label='T clustering',  edgecolors='white', linewidths=0.5)

        for dat, col in [(sp_data, COL_SP), (t_data, COL_T)]:
            x = dat[col_between].values
            y = dat['clustering_score'].values
            if len(x) > 1:
                m, b = np.polyfit(x, y, 1)
                xr = np.linspace(x.min(), x.max(), 100)
                ax.plot(xr, m*xr + b, color=col, linewidth=2, alpha=0.9)

        row = results[(results['phase']==phase) & (results['component']==comp)].iloc[0]
        t_main = row['t_between'];          p_main = row['pfdr_between']
        t_int  = row['t_between_x_clustT']; p_int  = row['pfdr_between_x_clustT']
        d_main = row['d_between']

        def pstr(p):
            if pd.isna(p):  return 'n.s.'
            if p < 0.001:   return 'p<.001'
            if p < 0.05:    return f'p={p:.3f}'
            return f'p={p:.2f} n.s.'

        ann = (f"Neural main: t={t_main:+.2f}, {pstr(p_main)}, d={d_main:.2f}\n"
               f"× Clust type: t={t_int:+.2f}, {pstr(p_int)}")
        ax.text(0.03, 0.97, ann, transform=ax.transAxes,
                fontsize=7.5, va='top', ha='left',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='#F7F7F7',
                          edgecolor='#CCCCCC', alpha=0.9))

        ax.text(-0.12, 1.05, panel_labels[pi][ci], transform=ax.transAxes,
                fontsize=14, fontweight='bold', va='top')

        comp_label  = 'Oscillatory' if comp == 'osc' else 'Aperiodic (Fractal)'
        phase_label = phase.capitalize()
        ax.set_title(f'{phase_label} — {comp_label} γ', fontsize=11, fontweight='bold', pad=8)
        ax.set_xlabel('Gamma power (subject mean, SSL)', fontsize=9)
        ax.set_ylabel('Clustering score', fontsize=9)

        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.grid(True, color=COL_GRID, linewidth=0.7, zorder=0)
        ax.tick_params(labelsize=8)

        if pi == 0 and ci == 0:
            ax.legend(fontsize=8.5, framealpha=0.9, loc='lower right')

fig.suptitle(
    'Between-subject gamma power predicts clustering\n'
    'across encoding and retrieval phases — Entorhinal Cortex (LEC/REC)',
    fontsize=13, fontweight='bold', y=1.01
)

plt.savefig('gamma_neural_x_clustering_type_EC.png',
            dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print("Figure saved: gamma_neural_x_clustering_type_EC.png")

# ============================================================================
# PRINT REPORT
# ============================================================================

lines = []
L = lines.append

L("=" * 110)
L("GAMMA-BAND iEEG LMM — Between- and Within-Subject Effects on Clustering  [LEC/REC VERSION]")
L("  Region factor: LEC (=0) vs REC (=1)")
L("  Fix 1: freq_hz removed from between model (unidentified at Level-2)")
L("  Fix 2: FDR correction (Benjamini-Hochberg) applied within each model level")
L("  Fix 3: Effect sizes reported — partial η²p and Cohen's d")
L("  Fix 4: Figure saved — gamma_neural_x_clustering_type_EC.png")
L("  Factors : clustering_type (SP=0/T=1), region (LEC=0/REC=1)")
L("  Sig     : ** uncorrected p<.05  | †† FDR-corrected p<.05")
L("=" * 110)

def fmt(label, t, p_unc, p_fdr, eta2, d=None):
    sig_unc = '*'  if (pd.notna(p_unc) and p_unc < 0.05) else ''
    sig_fdr = '†' if (pd.notna(p_fdr) and p_fdr < 0.05) else ''
    ts  = f'{t:+.3f}'    if pd.notna(t)     else '    NaN'
    pu  = f'{p_unc:.3f}' if pd.notna(p_unc) else '    NaN'
    pf  = f'{p_fdr:.3f}' if pd.notna(p_fdr) else '    NaN'
    e2  = f'{eta2:.3f}'  if pd.notna(eta2)  else '    NaN'
    d_s = f'{d:+.3f}'    if (d is not None and pd.notna(d)) else '      —'
    return (f"  {label:<40} {ts:>9} {pu:>9} {pf:>9} {e2:>8} {d_s:>9}  {sig_unc}{sig_fdr}")

HDR = f"  {'Effect':<40} {'t':>9} {'p_unc':>9} {'p_fdr':>9} {'η²p':>8} {'d':>9}  sig"
SEP = f"  {'─' * 95}"

for _, row in results.iterrows():
    phase = row['phase']; comp = row['component']
    L(f"\n{'━' * 110}")
    L(f"  PHASE: {phase.upper()}   COMPONENT: {'OSCILLATORY (OSC)' if comp=='osc' else 'APERIODIC/FRACTAL (FRAC)'}")
    L(f"{'━' * 110}")

    if row['note_between'] == 'ok':
        L(f"\n  BETWEEN-SUBJECT (Level-2 OLS)  "
          f"n_subjects={int(row['n_subjects'])}  n_obs={int(row['n_between'])}")
        L(f"  * uncorrected p<.05  † FDR-corrected p<.05")
        L(HDR); L(SEP)
        L(fmt('Neural main effect',         row['t_between'],         row['p_between'],         row['pfdr_between'],         row['eta2p_between'],         row['d_between']))
        L(fmt('Clustering type (T vs SP)',   row['t_clustT_between'],  row['p_clustT_between'],  row['pfdr_clustT_between'],  row['eta2p_clustT_between']))
        L(fmt('Region (REC vs LEC)',         row['t_region_between'],  row['p_region_between'],  row['pfdr_region_between'],  row['eta2p_region_between']))
        L(fmt('Neural × clustering_type',    row['t_between_x_clustT'],row['p_between_x_clustT'],row['pfdr_between_x_clustT'],row['eta2p_between_x_clustT']))
        L(fmt('Neural × region',             row['t_between_x_REC'],   row['p_between_x_REC'],   row['pfdr_between_x_REC'],   row['eta2p_between_x_REC']))
    else:
        L(f"\n  BETWEEN-SUBJECT  *** {row['note_between']} ***")

    status = row['note_within']
    if status.startswith('ok'):
        L(f"\n  WITHIN-SUBJECT (LMM: random slope + session nesting)  "
          f"n_obs={int(row['n_within'])}  [{status}]")
        L(f"  * uncorrected p<.05  † FDR-corrected p<.05")
        L(HDR); L(SEP)
        L(fmt('Neural main effect',         row['t_within'],          row['p_within'],          row['pfdr_within'],          row['eta2p_within'],          row['d_within']))
        L(fmt('Clustering type (T vs SP)',   row['t_clustT_within'],   row['p_clustT_within'],   row['pfdr_clustT_within'],   row['eta2p_clustT_within']))
        L(fmt('Region (REC vs LEC)',         row['t_region_within'],   row['p_region_within'],   row['pfdr_region_within'],   row['eta2p_region_within']))
        L(fmt('Frequency (freq_hz)',         row['t_freq_within'],     row['p_freq_within'],     row['pfdr_freq_within'],     row['eta2p_freq_within']))
        L(fmt('Neural × clustering_type',    row['t_within_x_clustT'], row['p_within_x_clustT'], row['pfdr_within_x_clustT'], row['eta2p_within_x_clustT']))
        L(fmt('Neural × region',             row['t_within_x_REC'],    row['p_within_x_REC'],    row['pfdr_within_x_REC'],    row['eta2p_within_x_REC']))
        L(fmt('Neural × freq_hz',            row['t_within_x_freq'],   row['p_within_x_freq'],   row['pfdr_within_x_freq'],   row['eta2p_within_x_freq']))
    else:
        L(f"\n  WITHIN-SUBJECT  *** {status} ***")

L(f"\n\n{'=' * 110}")
L("FDR CORRECTION SUMMARY")
L(f"{'=' * 110}")
L("Between-subject — effects surviving FDR correction:")
for _, row in results.iterrows():
    for pc, label in [('pfdr_between',         'Neural main'),
                      ('pfdr_between_x_clustT','Neural × clust_T'),
                      ('pfdr_between_x_REC',   'Neural × region'),
                      ('pfdr_clustT_between',  'Clust type main'),
                      ('pfdr_region_between',  'Region main')]:
        p = row.get(pc, np.nan)
        if pd.notna(p) and p < 0.05:
            L(f"  [{row['phase'].upper()} | {'OSC' if row['component']=='osc' else 'FRAC'}]  "
              f"{label:<30} p_fdr={p:.3f}")

L("\nWithin-subject — effects surviving FDR correction:")
found_within = False
for _, row in results.iterrows():
    for pc, label in [('pfdr_within',          'Neural main'),
                      ('pfdr_within_x_clustT', 'Neural × clust_T'),
                      ('pfdr_within_x_REC',    'Neural × region'),
                      ('pfdr_within_x_freq',   'Neural × freq_hz'),
                      ('pfdr_clustT_within',   'Clust type main'),
                      ('pfdr_region_within',   'Region main'),
                      ('pfdr_freq_within',     'Freq main')]:
        p = row.get(pc, np.nan)
        if pd.notna(p) and p < 0.05:
            found_within = True
            L(f"  [{row['phase'].upper()} | {'OSC' if row['component']=='osc' else 'FRAC'}]  "
              f"{label:<30} p_fdr={p:.3f}")
if not found_within:
    L("  None.")

L(f"\n{'=' * 110}")
L("FILES SAVED:")
L("  gamma_lmm_results_EC.csv                  — full results with FDR p-values and effect sizes")
L("  gamma_lmm_summary_EC.csv                  — FDR-significant effects only")
L("  gamma_neural_x_clustering_type_EC.png     — Figure: Neural × clustering_type (2×2 panel)")
L("=" * 110)

report = '\n'.join(lines)
print(report)

with open('gamma_lmm_report_EC.txt', 'w') as f:
    f.write(report)

sig_fdr_cols = [c for c in results.columns if c.startswith('sig_fdr_')]
results[results[sig_fdr_cols].any(axis=1)].to_csv('gamma_lmm_summary_EC.csv', index=False)

print("\nDone.")

Figure saved: gamma_neural_x_clustering_type_EC.png
GAMMA-BAND iEEG LMM — Between- and Within-Subject Effects on Clustering  [LEC/REC VERSION]
  Region factor: LEC (=0) vs REC (=1)
  Fix 1: freq_hz removed from between model (unidentified at Level-2)
  Fix 2: FDR correction (Benjamini-Hochberg) applied within each model level
  Fix 3: Effect sizes reported — partial η²p and Cohen's d
  Fix 4: Figure saved — gamma_neural_x_clustering_type_EC.png
  Factors : clustering_type (SP=0/T=1), region (LEC=0/REC=1)
  Sig     : ** uncorrected p<.05  | †† FDR-corrected p<.05

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  PHASE: ENCODING   COMPONENT: OSCILLATORY (OSC)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  BETWEEN-SUBJECT (Level-2 OLS)  n_subjects=16  n_obs=32
  * uncorrected p<.05  † FDR-corrected p<.05
  Effect                                           t    

In [16]:
"""
Gamma-band iEEG LMM Analysis — LPC/RPC Version
=============================================
Changes from EC version:
  - Region factor: LPC (=0) vs RPC (=1)
  - Filter: only rows where region in ['LPC', 'RPC']
  - All variable names, labels, and column references updated accordingly
"""

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# LOAD & FILTER TO GAMMA + LPC/RPC ONLY
# ============================================================================

df = pd.read_csv('ALL_SUBJECTS_irasa_per_freq-Copy1.csv')

gamma = df[df['freq_hz'] >= 40].copy()
gamma = gamma[gamma['region'].isin(['LPC', 'RPC'])].copy()
gamma = gamma[gamma['SP_clustering_from_prev'].notna()].copy()

# ============================================================================
# RESHAPE: clustering_type as factor
# ============================================================================

sp = gamma.copy()
sp['clustering_score'] = sp['SP_clustering_from_prev']
sp['clustering_type']  = 'SP'

t = gamma.copy()
t['clustering_score'] = t['T_clustering_from_prev']
t['clustering_type']  = 'T'

long = pd.concat([sp, t], ignore_index=True)
long['clust_T']    = (long['clustering_type'] == 'T').astype(int)
long['region_RPC'] = (long['region'] == 'RPC').astype(int)   # LPC=0, RPC=1
long['sess_id']    = long['subject'].astype(str) + '_' + long['session'].astype(str)

# ============================================================================
# BETWEEN / WITHIN DECOMPOSITION
# ============================================================================

for col in ['encoding_osc_ssl', 'encoding_frac_ssl',
            'retrieval_osc_ssl', 'retrieval_frac_ssl']:
    subj_mean              = long.groupby('subject')[col].transform('mean')
    long[col + '_between'] = subj_mean
    long[col + '_within']  = long[col] - subj_mean

# ============================================================================
# EFFECT SIZE HELPERS
# ============================================================================

def partial_eta_sq(t_val, df_resid):
    if pd.isna(t_val) or pd.isna(df_resid):
        return np.nan
    return t_val**2 / (t_val**2 + df_resid)

def cohens_d_from_t(t_val, n):
    if pd.isna(t_val) or pd.isna(n) or n == 0:
        return np.nan
    return t_val / np.sqrt(n)

# ============================================================================
# MODELS
# ============================================================================

PHASES = ['encoding', 'retrieval']
COMPS  = ['osc', 'frac']

results_rows = []

for phase in PHASES:
    for comp in COMPS:

        col_base    = f'{phase}_{comp}_ssl'
        col_between = col_base + '_between'
        col_within  = col_base + '_within'

        # ── BETWEEN-SUBJECT MODEL ─────────────────────────────────────────
        between_data = (
            long.groupby(['subject', 'clust_T', 'region_RPC'])
            [[col_between, 'clustering_score']]
            .mean()
            .reset_index()
            .dropna()
        )

        try:
            formula_b = (
                f'clustering_score ~ {col_between} + clust_T + region_RPC '
                f'+ {col_between}:clust_T '
                f'+ {col_between}:region_RPC'
            )
            res_b      = smf.ols(formula_b, data=between_data).fit()
            df_resid_b = res_b.df_resid
            n_subj     = between_data['subject'].nunique()

            def gb(key):
                t = res_b.tvalues.get(key, np.nan)
                p = res_b.pvalues.get(key, np.nan)
                e = partial_eta_sq(t, df_resid_b)
                return t, p, e

            t_b,         p_b,         e_b         = gb(col_between)
            t_b_xclust,  p_b_xclust,  e_b_xclust  = gb(f'{col_between}:clust_T')
            t_b_xregion, p_b_xregion, e_b_xregion = gb(f'{col_between}:region_RPC')
            t_clust_b,   p_clust_b,   e_clust_b   = gb('clust_T')
            t_region_b,  p_region_b,  e_region_b  = gb('region_RPC')
            d_b  = cohens_d_from_t(t_b, n_subj)
            n_b  = int(res_b.nobs)
            note_b = 'ok'

        except Exception as e:
            (t_b, p_b, e_b, t_b_xclust, p_b_xclust, e_b_xclust,
             t_b_xregion, p_b_xregion, e_b_xregion,
             t_clust_b, p_clust_b, e_clust_b,
             t_region_b, p_region_b, e_region_b,
             d_b, n_b, n_subj) = [np.nan] * 18
            note_b = f'failed: {e}'

        # ── WITHIN-SUBJECT MODEL (LMM) ────────────────────────────────────
        within_data = long[[
            'subject', 'sess_id', 'clustering_score',
            col_within, 'clust_T', 'region_RPC', 'freq_hz'
        ]].dropna().copy()
        within_data = within_data.rename(columns={col_within: 'neural_within'})

        try:
            formula_w = (
                'clustering_score ~ neural_within + clust_T + region_RPC + freq_hz '
                '+ neural_within:clust_T '
                '+ neural_within:region_RPC '
                '+ neural_within:freq_hz'
            )
            md = smf.mixedlm(
                formula_w,
                data=within_data,
                groups=within_data['subject'],
                re_formula='~neural_within',
                vc_formula={'sess_id': '0 + C(sess_id)'}
            )
            res_w      = md.fit(reml=True, method='powell')
            df_resid_w = res_w.df_resid

            def gw(key):
                t = res_w.tvalues.get(key, np.nan)
                p = res_w.pvalues.get(key, np.nan)
                e = partial_eta_sq(t, df_resid_w)
                return t, p, e

            t_w,         p_w,         e_w         = gw('neural_within')
            t_w_xclust,  p_w_xclust,  e_w_xclust  = gw('neural_within:clust_T')
            t_w_xregion, p_w_xregion, e_w_xregion = gw('neural_within:region_RPC')
            t_w_xfreq,   p_w_xfreq,   e_w_xfreq   = gw('neural_within:freq_hz')
            t_clust_w,   p_clust_w,   e_clust_w   = gw('clust_T')
            t_region_w,  p_region_w,  e_region_w  = gw('region_RPC')
            t_freq_w,    p_freq_w,    e_freq_w    = gw('freq_hz')
            d_w    = cohens_d_from_t(t_w, within_data['subject'].nunique())
            n_w    = int(res_w.nobs)
            conv_w = res_w.converged
            note_w = 'ok' if conv_w else 'ok (convergence warning)'

        except Exception as e:
            (t_w, p_w, e_w, t_w_xclust, p_w_xclust, e_w_xclust,
             t_w_xregion, p_w_xregion, e_w_xregion,
             t_w_xfreq, p_w_xfreq, e_w_xfreq,
             t_clust_w, p_clust_w, e_clust_w,
             t_region_w, p_region_w, e_region_w,
             t_freq_w, p_freq_w, e_freq_w,
             d_w, n_w) = [np.nan] * 23
            note_w = f'failed: {e}'

        results_rows.append({
            'phase': phase, 'component': comp,
            # Between
            't_between':          t_b,         'p_between':          p_b,         'eta2p_between':          e_b,
            't_between_x_clustT': t_b_xclust,  'p_between_x_clustT': p_b_xclust,  'eta2p_between_x_clustT': e_b_xclust,
            't_between_x_RPC':    t_b_xregion, 'p_between_x_RPC':    p_b_xregion, 'eta2p_between_x_RPC':    e_b_xregion,
            't_clustT_between':   t_clust_b,   'p_clustT_between':   p_clust_b,   'eta2p_clustT_between':   e_clust_b,
            't_region_between':   t_region_b,  'p_region_between':   p_region_b,  'eta2p_region_between':   e_region_b,
            'd_between':          d_b,
            'n_between':          n_b,         'n_subjects':         n_subj,      'note_between': note_b,
            # Within
            't_within':           t_w,         'p_within':           p_w,         'eta2p_within':           e_w,
            't_within_x_clustT':  t_w_xclust,  'p_within_x_clustT':  p_w_xclust,  'eta2p_within_x_clustT':  e_w_xclust,
            't_within_x_RPC':     t_w_xregion, 'p_within_x_RPC':     p_w_xregion, 'eta2p_within_x_RPC':     e_w_xregion,
            't_within_x_freq':    t_w_xfreq,   'p_within_x_freq':    p_w_xfreq,   'eta2p_within_x_freq':    e_w_xfreq,
            't_clustT_within':    t_clust_w,   'p_clustT_within':    p_clust_w,   'eta2p_clustT_within':    e_clust_w,
            't_region_within':    t_region_w,  'p_region_within':    p_region_w,  'eta2p_region_within':    e_region_w,
            't_freq_within':      t_freq_w,    'p_freq_within':      p_freq_w,    'eta2p_freq_within':      e_freq_w,
            'd_within':           d_w,
            'n_within':           n_w,         'note_within':        note_w,
        })

results = pd.DataFrame(results_rows)

# ============================================================================
# FDR CORRECTION (Benjamini-Hochberg)
# ============================================================================

def apply_fdr(results, p_cols, suffix):
    all_p = []
    for pc in p_cols:
        all_p.extend(results[pc].tolist())
    all_p = np.array(all_p, dtype=float)
    valid_mask = ~np.isnan(all_p)
    fdr_p = np.full_like(all_p, np.nan)
    if valid_mask.sum() > 0:
        _, p_corrected, _, _ = multipletests(all_p[valid_mask], method='fdr_bh')
        fdr_p[valid_mask] = p_corrected
    idx = 0
    for pc in p_cols:
        n = len(results)
        col_fdr = pc.replace('p_', 'pfdr_')
        col_sig = pc.replace('p_', 'sig_fdr_')
        results[col_fdr] = fdr_p[idx:idx+n]
        results[col_sig] = results[col_fdr].apply(
            lambda x: x < 0.05 if pd.notna(x) else False)
        idx += n
    return results

between_p_cols = ['p_between', 'p_between_x_clustT', 'p_between_x_RPC',
                  'p_clustT_between', 'p_region_between']
within_p_cols  = ['p_within', 'p_within_x_clustT', 'p_within_x_RPC',
                  'p_within_x_freq', 'p_clustT_within', 'p_region_within', 'p_freq_within']

results = apply_fdr(results, between_p_cols, 'between')
results = apply_fdr(results, within_p_cols,  'within')

for pc in between_p_cols + within_p_cols:
    results['sig_unc_' + pc[2:]] = results[pc].apply(
        lambda x: x < 0.05 if pd.notna(x) else False)

# ============================================================================
# SAVE RESULTS
# ============================================================================

results.to_csv('gamma_lmm_results_PC.csv', index=False)

# ============================================================================
# VISUALIZATION
# ============================================================================

def get_between_plot_data(long, col_between):
    data = (
        long.groupby(['subject', 'clust_T'])
        [[col_between, 'clustering_score']]
        .mean()
        .reset_index()
        .dropna()
    )
    sp_data = data[data['clust_T'] == 0].copy()
    t_data  = data[data['clust_T'] == 1].copy()
    return sp_data, t_data

COL_SP   = '#2166AC'
COL_T    = '#D6604D'
COL_GRID = '#E8E8E8'

fig = plt.figure(figsize=(12, 10))
fig.patch.set_facecolor('white')
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.38)

panel_labels = [['A', 'B'], ['C', 'D']]

for pi, phase in enumerate(PHASES):
    for ci, comp in enumerate(COMPS):

        ax = fig.add_subplot(gs[pi, ci])
        col_base    = f'{phase}_{comp}_ssl'
        col_between = col_base + '_between'

        sp_data, t_data = get_between_plot_data(long, col_between)

        ax.scatter(sp_data[col_between], sp_data['clustering_score'],
                   color=COL_SP, s=60, zorder=3, alpha=0.85,
                   label='SP clustering', edgecolors='white', linewidths=0.5)
        ax.scatter(t_data[col_between],  t_data['clustering_score'],
                   color=COL_T,  s=60, zorder=3, alpha=0.85,
                   label='T clustering',  edgecolors='white', linewidths=0.5)

        for dat, col in [(sp_data, COL_SP), (t_data, COL_T)]:
            x = dat[col_between].values
            y = dat['clustering_score'].values
            if len(x) > 1:
                m, b = np.polyfit(x, y, 1)
                xr = np.linspace(x.min(), x.max(), 100)
                ax.plot(xr, m*xr + b, color=col, linewidth=2, alpha=0.9)

        row = results[(results['phase']==phase) & (results['component']==comp)].iloc[0]
        t_main = row['t_between'];          p_main = row['pfdr_between']
        t_int  = row['t_between_x_clustT']; p_int  = row['pfdr_between_x_clustT']
        d_main = row['d_between']

        def pstr(p):
            if pd.isna(p):  return 'n.s.'
            if p < 0.001:   return 'p<.001'
            if p < 0.05:    return f'p={p:.3f}'
            return f'p={p:.2f} n.s.'

        ann = (f"Neural main: t={t_main:+.2f}, {pstr(p_main)}, d={d_main:.2f}\n"
               f"× Clust type: t={t_int:+.2f}, {pstr(p_int)}")
        ax.text(0.03, 0.97, ann, transform=ax.transAxes,
                fontsize=7.5, va='top', ha='left',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='#F7F7F7',
                          edgecolor='#CCCCCC', alpha=0.9))

        ax.text(-0.12, 1.05, panel_labels[pi][ci], transform=ax.transAxes,
                fontsize=14, fontweight='bold', va='top')

        comp_label  = 'Oscillatory' if comp == 'osc' else 'Aperiodic (Fractal)'
        phase_label = phase.capitalize()
        ax.set_title(f'{phase_label} — {comp_label} γ', fontsize=11, fontweight='bold', pad=8)
        ax.set_xlabel('Gamma power (subject mean, SSL)', fontsize=9)
        ax.set_ylabel('Clustering score', fontsize=9)

        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.grid(True, color=COL_GRID, linewidth=0.7, zorder=0)
        ax.tick_params(labelsize=8)

        if pi == 0 and ci == 0:
            ax.legend(fontsize=8.5, framealpha=0.9, loc='lower right')

fig.suptitle(
    'Between-subject gamma power predicts clustering\n'
    'across encoding and retrieval phases — Parietal Cortex (LPC/RPC)',
    fontsize=13, fontweight='bold', y=1.01
)

plt.savefig('gamma_neural_x_clustering_type_PC.png',
            dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print("Figure saved: gamma_neural_x_clustering_type_PC.png")

# ============================================================================
# PRINT REPORT
# ============================================================================

lines = []
L = lines.append

L("=" * 110)
L("GAMMA-BAND iEEG LMM — Between- and Within-Subject Effects on Clustering  [LPC/RPC VERSION]")
L("  Region factor: LPC (=0) vs RPC (=1)")
L("  Fix 1: freq_hz removed from between model (unidentified at Level-2)")
L("  Fix 2: FDR correction (Benjamini-Hochberg) applied within each model level")
L("  Fix 3: Effect sizes reported — partial η²p and Cohen's d")
L("  Fix 4: Figure saved — gamma_neural_x_clustering_type_PC.png")
L("  Factors : clustering_type (SP=0/T=1), region (LPC=0/RPC=1)")
L("  Sig     : ** uncorrected p<.05  | †† FDR-corrected p<.05")
L("=" * 110)

def fmt(label, t, p_unc, p_fdr, eta2, d=None):
    sig_unc = '*'  if (pd.notna(p_unc) and p_unc < 0.05) else ''
    sig_fdr = '†' if (pd.notna(p_fdr) and p_fdr < 0.05) else ''
    ts  = f'{t:+.3f}'    if pd.notna(t)     else '    NaN'
    pu  = f'{p_unc:.3f}' if pd.notna(p_unc) else '    NaN'
    pf  = f'{p_fdr:.3f}' if pd.notna(p_fdr) else '    NaN'
    e2  = f'{eta2:.3f}'  if pd.notna(eta2)  else '    NaN'
    d_s = f'{d:+.3f}'    if (d is not None and pd.notna(d)) else '      —'
    return (f"  {label:<40} {ts:>9} {pu:>9} {pf:>9} {e2:>8} {d_s:>9}  {sig_unc}{sig_fdr}")

HDR = f"  {'Effect':<40} {'t':>9} {'p_unc':>9} {'p_fdr':>9} {'η²p':>8} {'d':>9}  sig"
SEP = f"  {'─' * 95}"

for _, row in results.iterrows():
    phase = row['phase']; comp = row['component']
    L(f"\n{'━' * 110}")
    L(f"  PHASE: {phase.upper()}   COMPONENT: {'OSCILLATORY (OSC)' if comp=='osc' else 'APERIODIC/FRACTAL (FRAC)'}")
    L(f"{'━' * 110}")

    if row['note_between'] == 'ok':
        L(f"\n  BETWEEN-SUBJECT (Level-2 OLS)  "
          f"n_subjects={int(row['n_subjects'])}  n_obs={int(row['n_between'])}")
        L(f"  * uncorrected p<.05  † FDR-corrected p<.05")
        L(HDR); L(SEP)
        L(fmt('Neural main effect',         row['t_between'],         row['p_between'],         row['pfdr_between'],         row['eta2p_between'],         row['d_between']))
        L(fmt('Clustering type (T vs SP)',   row['t_clustT_between'],  row['p_clustT_between'],  row['pfdr_clustT_between'],  row['eta2p_clustT_between']))
        L(fmt('Region (RPC vs LPC)',         row['t_region_between'],  row['p_region_between'],  row['pfdr_region_between'],  row['eta2p_region_between']))
        L(fmt('Neural × clustering_type',    row['t_between_x_clustT'],row['p_between_x_clustT'],row['pfdr_between_x_clustT'],row['eta2p_between_x_clustT']))
        L(fmt('Neural × region',             row['t_between_x_RPC'],   row['p_between_x_RPC'],   row['pfdr_between_x_RPC'],   row['eta2p_between_x_RPC']))
    else:
        L(f"\n  BETWEEN-SUBJECT  *** {row['note_between']} ***")

    status = row['note_within']
    if status.startswith('ok'):
        L(f"\n  WITHIN-SUBJECT (LMM: random slope + session nesting)  "
          f"n_obs={int(row['n_within'])}  [{status}]")
        L(f"  * uncorrected p<.05  † FDR-corrected p<.05")
        L(HDR); L(SEP)
        L(fmt('Neural main effect',         row['t_within'],          row['p_within'],          row['pfdr_within'],          row['eta2p_within'],          row['d_within']))
        L(fmt('Clustering type (T vs SP)',   row['t_clustT_within'],   row['p_clustT_within'],   row['pfdr_clustT_within'],   row['eta2p_clustT_within']))
        L(fmt('Region (RPC vs LPC)',         row['t_region_within'],   row['p_region_within'],   row['pfdr_region_within'],   row['eta2p_region_within']))
        L(fmt('Frequency (freq_hz)',         row['t_freq_within'],     row['p_freq_within'],     row['pfdr_freq_within'],     row['eta2p_freq_within']))
        L(fmt('Neural × clustering_type',    row['t_within_x_clustT'], row['p_within_x_clustT'], row['pfdr_within_x_clustT'], row['eta2p_within_x_clustT']))
        L(fmt('Neural × region',             row['t_within_x_RPC'],    row['p_within_x_RPC'],    row['pfdr_within_x_RPC'],    row['eta2p_within_x_RPC']))
        L(fmt('Neural × freq_hz',            row['t_within_x_freq'],   row['p_within_x_freq'],   row['pfdr_within_x_freq'],   row['eta2p_within_x_freq']))
    else:
        L(f"\n  WITHIN-SUBJECT  *** {status} ***")

L(f"\n\n{'=' * 110}")
L("FDR CORRECTION SUMMARY")
L(f"{'=' * 110}")
L("Between-subject — effects surviving FDR correction:")
for _, row in results.iterrows():
    for pc, label in [('pfdr_between',         'Neural main'),
                      ('pfdr_between_x_clustT','Neural × clust_T'),
                      ('pfdr_between_x_RPC',   'Neural × region'),
                      ('pfdr_clustT_between',  'Clust type main'),
                      ('pfdr_region_between',  'Region main')]:
        p = row.get(pc, np.nan)
        if pd.notna(p) and p < 0.05:
            L(f"  [{row['phase'].upper()} | {'OSC' if row['component']=='osc' else 'FRAC'}]  "
              f"{label:<30} p_fdr={p:.3f}")

L("\nWithin-subject — effects surviving FDR correction:")
found_within = False
for _, row in results.iterrows():
    for pc, label in [('pfdr_within',          'Neural main'),
                      ('pfdr_within_x_clustT', 'Neural × clust_T'),
                      ('pfdr_within_x_RPC',    'Neural × region'),
                      ('pfdr_within_x_freq',   'Neural × freq_hz'),
                      ('pfdr_clustT_within',   'Clust type main'),
                      ('pfdr_region_within',   'Region main'),
                      ('pfdr_freq_within',     'Freq main')]:
        p = row.get(pc, np.nan)
        if pd.notna(p) and p < 0.05:
            found_within = True
            L(f"  [{row['phase'].upper()} | {'OSC' if row['component']=='osc' else 'FRAC'}]  "
              f"{label:<30} p_fdr={p:.3f}")
if not found_within:
    L("  None.")

L(f"\n{'=' * 110}")
L("FILES SAVED:")
L("  gamma_lmm_results_PC.csv                  — full results with FDR p-values and effect sizes")
L("  gamma_lmm_summary_PC.csv                  — FDR-significant effects only")
L("  gamma_neural_x_clustering_type_PC.png     — Figure: Neural × clustering_type (2×2 panel)")
L("=" * 110)

report = '\n'.join(lines)
print(report)

with open('gamma_lmm_report_PC.txt', 'w') as f:
    f.write(report)

sig_fdr_cols = [c for c in results.columns if c.startswith('sig_fdr_')]
results[results[sig_fdr_cols].any(axis=1)].to_csv('gamma_lmm_summary_PC.csv', index=False)

print("\nDone.")

Figure saved: gamma_neural_x_clustering_type_PC.png
GAMMA-BAND iEEG LMM — Between- and Within-Subject Effects on Clustering  [LPC/RPC VERSION]
  Region factor: LPC (=0) vs RPC (=1)
  Fix 1: freq_hz removed from between model (unidentified at Level-2)
  Fix 2: FDR correction (Benjamini-Hochberg) applied within each model level
  Fix 3: Effect sizes reported — partial η²p and Cohen's d
  Fix 4: Figure saved — gamma_neural_x_clustering_type_PC.png
  Factors : clustering_type (SP=0/T=1), region (LPC=0/RPC=1)
  Sig     : ** uncorrected p<.05  | †† FDR-corrected p<.05

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  PHASE: ENCODING   COMPONENT: OSCILLATORY (OSC)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  BETWEEN-SUBJECT (Level-2 OLS)  n_subjects=16  n_obs=32
  * uncorrected p<.05  † FDR-corrected p<.05
  Effect                                           t    